# Test domain samples — 97 URL / 97 hostname

Chạy các ô từ trên xuống (**Run All**). Kaggle cần bật **Internet**; chỉ dùng CPU.
Notebook chọn một URL gốc cho mỗi hostname,
lấy từ `links_corpus.parquet` (4.394.718 dòng; seed 42).
Có 97 mẫu cũ và 0 mẫu bổ sung.
Không cần clone repo, tải Parquet hay upload code Python.

Dùng snapshot crawler và probe hiện tại: tải trực tiếp URL mẫu, retry,
giới hạn tốc độ, tải HTML/PDF, trích xuất và kiểm tra chất lượng (QC).
Kết quả sẽ khác theo thời điểm và môi trường mạng.
HTTP User-Agent là Mozilla/Chrome; Accept và Accept-Language được đặt theo trình duyệt.
User-Agent không đảm bảo hết 403; kết quả thực tế được giữ nguyên.
Bảng ưu tiên domain lỗi xếp theo số URL hostname trong toàn dataset giảm dần,
không xếp theo số lỗi mẫu. Giữ riêng subdomain, số dòng nguồn chưa loại URL trùng.

- **VAO_DUOC**: URL cuối trả HTTP 2xx, kể cả khi trích xuất/QC thất bại.
- **KHONG_VAO_DUOC**: đã chạy nhưng không lấy được HTTP 2xx của trang đích.
- **CHUA_CHAY**: chưa có checkpoint, không tính là lỗi.

`robots_http_status` là HTTP của robots.txt; `http_status` là HTTP của trang.
Dừng do robots là kết quả của chính sách crawler, không chứng minh website
không mở được bằng trình duyệt. Nhóm ưu tiên có thêm đối chiếu Selenium.
Vài mẫu mỗi hostname chưa đại diện cho toàn bộ URL trên website.


Toàn bộ **97 hostname**, mỗi hostname một URL; giữ bộ mẫu reservoir seed 42 ban đầu.

            Chế độ này tải trực tiếp URL mẫu công khai bằng Requests với header trình duyệt;
            robots không chặn bước kiểm tra truy cập. Kết quả Requests và Selenium được giữ riêng.
            `BROWSER_TEST_MODE` mặc định thử Selenium cho URL Requests lỗi, trang JS, text rỗng hoặc QC lỗi;
            `first_each_domain_and_failures` dùng khi muốn đối chiếu thêm mẫu đầu mỗi hostname.
            HTTP 2xx vẫn cần QC: trang lỗi trả 200 hoặc nội dung quảng cáo không phải bài sạch.


## 1. Cấu hình

Mặc định chạy toàn bộ URL đã nhúng. Có thể đặt `MAX_NEW = 10` để thử trước, rồi đổi
thành `None` và chỉ chạy lại ô crawl để làm tiếp. Checkpoint đã có, kể cả
lỗi, được bỏ qua. Muốn thử lại tất cả URL từ đầu, chạy lại ô cấu hình để
tạo thư mục mới. `RESUME_RUN_DIR` dùng để tiếp tục thư mục output đã có.
Code mới không resume trực tiếp run dùng snapshot cũ: đặt `RESUME_RUN_DIR = None`.
Có thể đặt `RECOVERY_RUN_DIR` trỏ tới run cũ đã giải nén để trích lại raw hợp lệ
bằng parser mới và tải lại URL lỗi trong một run mới.


In [ ]:
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

WORKERS = 8
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
NETWORK_CHECK_DOMAINS = ('www.120ask.com', 'bingli.iiyi.com', 'article.iiyi.com')
BROWSER_TEST_MODE = 'failed_only'  # off / first_each_domain_and_failures / all
BROWSER_USER_AGENT = None  # None: dùng User-Agent thực của Chrome đang chạy
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None       # None: Selenium Manager tìm hoặc tải Chrome phù hợp
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
PAGE_ATTEMPTS = 3
MAX_NEW = None           # None: toàn bộ; 10: thêm tối đa 10 URL
RESUME_RUN_DIR = None    # Hoặc đường dẫn thư mục run đã có
RECOVERY_RUN_DIR = None  # Run cũ: tạo run mới, reuse raw 2xx và thử lại các URL lỗi

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import uuid

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python 3.11 trở lên.')
if type(WORKERS) is not int or WORKERS < 1:
    raise ValueError('WORKERS phải là số nguyên dương.')
if type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('ATTEMPTS phải là số nguyên dương.')
if isinstance(DELAY, bool) or not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if RECOVERY_RUN_DIR and RESUME_RUN_DIR:
    raise ValueError('Recovery tạo run mới; không đặt đồng thời RESUME_RUN_DIR.')
OUTPUT_BASE = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
               else Path('/content') if Path('/content').is_dir()
               else Path.cwd() / 'runs')
RUN_DIR = (Path(RESUME_RUN_DIR).expanduser().resolve() if RESUME_RUN_DIR else
           OUTPUT_BASE / ('domain_probe_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
                          + '_' + uuid.uuid4().hex[:6]))
if RESUME_RUN_DIR and not (RUN_DIR / 'manifest.json').is_file():
    raise ValueError('RESUME_RUN_DIR phải chứa manifest.json của notebook này.')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', RUN_DIR)

DIRECT_PUBLIC_PROBE = True
NETWORK_CHECK_DOMAINS = ('www.cnkang.com', 'www.120ask.com', 'www.familydoctor.com.cn', 'ask.39.net', 'zysjonline.com', 'www.a-hospital.com', 'suckhoecongdongonline.vn', 'zhongyibaodian.net', 'suckhoedoisong.vn', 'nhathuoclongchau.com.vn', 'www.zydcd.com', 'thanhnien.vn', 'www.wujue.com', 'www.youlai.cn', 'laodong.vn', 'baby.39.net', 'phunusuckhoe.giadinhonline.vn', 'www.vinmec.com', 'bingli.iiyi.com', 'medlatec.vn', 'qihuangzhishu.com', 'jbk.39.net', 'giadinhonline.vn', 'baohaiphong.vn', 'baonghean.vn', 'vietnamnet.vn', 'care.39.net', 'test.pmphai.com', 'baodanang.vn', 'tiemchunglongchau.com.vn', 'fitness.39.net', 'baocantho.com.vn', 'hellobacsi.com', 'woman.39.net', 'food.39.net', 'baoangiang.com.vn', 'fk.39.net', 'vov.vn', 'cancer.39.net', 'khoahocphothong.vn', 'jb39.com', 'youmed.vn', 'vov2.vov.vn', 'nk.39.net', 'phuyen.baodaklak.vn', 'www.msdmanuals.cn', 'heart.39.net', 'article.iiyi.com', 'suckhoeviet.org.vn', 'baoquangtri.vn', 'pmc-ecm-healthblog.beta.pharmacity.io', 'man.39.net', 'nhandan.vn', 'baovinhlong.com.vn', 'gan.39.net', 'tamanhhospital.vn', 'tnb.39.net', 'wei.39.net', 'tuoitre.vn', 'baidianfeng.familydoctor.com.cn', 'www.qdnd.vn', 'www.vietnamplus.vn', 'baogialai.com.vn', 'shen.39.net', 'ek.39.net', 'tienphong.vn', 'thaythuocvietnam.vn', 'baoquangninh.vn', 'bachmai.gov.vn', 'www.baidu.com', 'baochinhphu.vn', 'dantri.com.vn', 'gk.39.net', 'vnexpress.net', 'baolangson.vn', 'www.sggp.org.vn', 'gc.39.net', 'pf.39.net', 'health.people.com.cn', 'baophutho.vn', 'hanoimoi.vn', 'benhviennhitrunguong.gov.vn', 'baotayninh.vn', 'sj.39.net', 'baothanhhoa.vn', 'byby.39.net', 'benhvienvietduc.org', 'www.pharmacity.vn', 'www.familydoctor.cn', 'qy.familydoctor.com.cn', 'v.familydoctor.com.cn', 'ask.familydoctor.com.cn', 'mega.vietnamplus.vn', 'yanglao.familydoctor.com.cn', 'fk.99.com.cn', 'special.vietnamplus.vn', 'ypk.familydoctor.com.cn')


## 2. Cài dependencies

Cài các package crawler cần trong môi trường hiện tại. Phiên bản thực tế
và log pip được lưu trong output. Khi resume, dùng phiên bản đã lưu.


In [ ]:
import subprocess
import importlib.metadata

packages = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'ipython']
if DIRECT_PUBLIC_PROBE and BROWSER_TEST_MODE != 'off':
    packages.append('selenium')
requirement_file = RUN_DIR / 'notebook_requirements.txt'
install_args = ['-r', str(requirement_file)] if requirement_file.exists() else packages
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding='utf-8', errors='replace',
)
(RUN_DIR / 'pip_install.log').write_text(installation.stdout or '', encoding='utf-8')
if installation.returncode:
    print(installation.stdout)
    raise RuntimeError('Cài dependencies thất bại; xem pip_install.log.')
versions = {name: importlib.metadata.version(name) for name in packages}
if not requirement_file.exists():
    requirement_file.write_text(''.join(f'{p}=={v}\n' for p, v in versions.items()), encoding='utf-8')
print(json.dumps(versions, indent=2))


## 3. Danh sách mẫu và snapshot crawler

97 mẫu giữ nguyên ID/URL nguồn. Ô code có thể mở để kiểm tra danh sách
và snapshot. Crawler được kiểm tra SHA-256 trước khi import.


In [ ]:
SAMPLES = [{'id': 1826763,
  'url': 'http://www.cnkang.com/myjk/201202/480566.html',
  'domain': 'www.cnkang.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2843944,
  'url': 'https://www.120ask.com/question/38159402.htm',
  'domain': 'www.120ask.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3891869,
  'url': 'https://www.familydoctor.com.cn/tangniaobing/a/201702/1706297.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2211756,
  'url': 'https://ask.39.net/question/20931974.html',
  'domain': 'ask.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4182981,
  'url': 'https://zysjonline.com/articles/misc/16204/',
  'domain': 'zysjonline.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2992495,
  'url': 'https://www.a-hospital.com/w/%E4%B8%8A%E8%85%94%E9%9D%99%E8%84%89%E7%BB%BC%E5%90%88%E5%BE%81',
  'domain': 'www.a-hospital.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 459741,
  'url': 'https://suckhoecongdongonline.vn/tu-vi-thang-5-2021-am-lich-cua-12-con-giap-chi-tiet-nhat',
  'domain': 'suckhoecongdongonline.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4065823,
  'url': 'https://zhongyibaodian.net/a/142698.html',
  'domain': 'zhongyibaodian.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 546923,
  'url': 'https://suckhoedoisong.vn/tai-sao-ngay-nao-cung-can-chong-nang-cho-da-16995354.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 268030,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/thong-tin-can-biet-ve-phuong-phap-phau-thuat-lay-duong-ro-luan-nhi-59283.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2065346,
  'url': 'http://www.zydcd.com/zhongyi/zy20700zh/',
  'domain': 'www.zydcd.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 651969,
  'url': 'https://thanhnien.vn/tong-hoi-y-hoc-vn-de-nghi-cho-bac-si-vu-tai-bien-chay-than-duoc-tai-ngoai-185676141.htm',
  'domain': 'thanhnien.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2040931,
  'url': 'http://www.wujue.com/zyzy/zycd/p/200704/14378.html',
  'domain': 'www.wujue.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4013480,
  'url': 'https://www.youlai.cn/yyk/article/608596.html',
  'domain': 'www.youlai.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 154292,
  'url': 'https://laodong.vn/suc-khoe/4-thuc-uong-giai-nhiet-giup-giam-trao-nguoc-axit-1205578.ldo',
  'domain': 'laodong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 772903,
  'url': 'http://baby.39.net/a/141222/4540983.html',
  'domain': 'baby.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 304343,
  'url': 'https://phunusuckhoe.giadinhonline.vn/nhung-loi-ich-bat-ngo-cua-nam-oi-voi-nguoi-benh-tim-mach-c12a511869.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 750899,
  'url': 'https://www.vinmec.com/vie/bai-viet/roi-loan-tieu-tien-o-phu-nu-mang-thai-vi',
  'domain': 'www.vinmec.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2501085,
  'url': 'https://bingli.iiyi.com/show/64998-1.html',
  'domain': 'bingli.iiyi.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 176969,
  'url': 'https://medlatec.vn/tin-tuc/bi-mat-cam-xuc-khi-quan-he-co-nguyen-nhan-la-do-dau-s78-n22073',
  'domain': 'medlatec.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 895407,
  'url': 'http://qihuangzhishu.com/764/120.htm',
  'domain': 'qihuangzhishu.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2525727,
  'url': 'https://jbk.39.net/shoushu/xws/',
  'domain': 'jbk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 108128,
  'url': 'https://giadinhonline.vn/giup-me-bau-ngu-ngon-voi-10-meo-don-gian-d19025.html',
  'domain': 'giadinhonline.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 58905,
  'url': 'https://baohaiphong.vn/tat-ca-benh-nhan-covid-19-cua-hai-duong-deu-da-xuat-vien-353669.html',
  'domain': 'baohaiphong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 67234,
  'url': 'https://baonghean.vn/an-uong-lam-am-co-the-10014055.html',
  'domain': 'baonghean.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 686659,
  'url': 'https://vietnamnet.vn/cach-dap-mat-na-cham-soc-da-mat-phu-hop-voi-tung-loai-da-2018837.html',
  'domain': 'vietnamnet.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 801009,
  'url': 'http://care.39.net/a/170224/5230598.html',
  'domain': 'care.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2541313,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=0001AA100000000EO44K',
  'domain': 'test.pmphai.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 34798,
  'url': 'https://baodanang.vn/khong-chu-quan-voi-benh-glaucoma-3239403.html',
  'domain': 'baodanang.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 674464,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/vi-khuan-nhuom-soi-la-gi-y-nghia-trong-chan-doan-nhiem-trung',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 818281,
  'url': 'http://fitness.39.net/a/180124/6017569.html',
  'domain': 'fitness.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 23544,
  'url': 'https://baocantho.com.vn/tai-sao-chung-ta-e-oc-moi-khi-cao-mong-tay-len-bang-a72145.html',
  'domain': 'baocantho.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 125814,
  'url': 'https://hellobacsi.com/nuoi-day-con/nhi-khoa/van-de-nhi-khoa-khac/tre-co-ham-rang-bi-o-vang/',
  'domain': 'hellobacsi.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 914722,
  'url': 'http://woman.39.net/a/140915/4472540.html',
  'domain': 'woman.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 844242,
  'url': 'http://food.39.net/a/2010821/1451769.html',
  'domain': 'food.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 13933,
  'url': 'https://baoangiang.com.vn/who-canh-bao-nguy-co-lay-lan-cao-cua-virus-chikungunya-a424815.html',
  'domain': 'baoangiang.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 836246,
  'url': 'http://fk.39.net/a/241010/i1trnyn.html',
  'domain': 'fk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 710247,
  'url': 'https://vov.vn/suc-khoe/nhung-thu-co-the-can-khi-ban-gia-di-post981896.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2507808,
  'url': 'https://cancer.39.net/a/210407/8822699.html',
  'domain': 'cancer.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 140282,
  'url': 'https://khoahocphothong.vn/phat-hien-khang-the-co-the-ngan-ngua-hau-het-cac-chung-hiv-240586.html',
  'domain': 'khoahocphothong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2516720,
  'url': 'https://jb39.com/jibing/ZiGongNeiMoZengSheng264317.htm',
  'domain': 'jb39.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 760817,
  'url': 'https://youmed.vn/tin-tuc/chi-nha-khoa-va-tam-nuoc-dung-cu-nao-tot-hon/',
  'domain': 'youmed.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 717975,
  'url': 'https://vov2.vov.vn/suc-khoe/phong-va-dieu-tri-benh-viem-bo-mi-mat-24968.vov2',
  'domain': 'vov2.vov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 871353,
  'url': 'http://nk.39.net/a/150617/4640167.html',
  'domain': 'nk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 312382,
  'url': 'https://phuyen.baodaklak.vn/95/141195/mit-tinh-huong-ung-tuan-le-truyen-thong-ve-phong-chong-khang-thuoc/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3980742,
  'url': 'https://www.msdmanuals.cn/home/skin-disorders/nail-disorders',
  'domain': 'www.msdmanuals.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 860914,
  'url': 'http://heart.39.net/a/190801/7350145.html',
  'domain': 'heart.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2136074,
  'url': 'https://article.iiyi.com/detail/416303.html',
  'domain': 'article.iiyi.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 565289,
  'url': 'https://suckhoeviet.org.vn/cong-dung-tuyet-voi-cua-hoa-dau-biec-voi-suc-khoe-va-nhung-luu-y-khi-su-dung-8483.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 88645,
  'url': 'https://baoquangtri.vn/suc-khoe/202207/50-benh-nhan-covid-19-dang-dieu-tri-tai-benh-vien-2202255/',
  'domain': 'baoquangtri.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 319666,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/hoat-chat-amitriptylin/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 868618,
  'url': 'http://man.39.net/a/220924/8244035.html',
  'domain': 'man.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 202493,
  'url': 'https://nhandan.vn/post-853313.html',
  'domain': 'nhandan.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 97029,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/202511/dung-lieu-an-thuc-pham-moc-nguy-hiem-khon-luong-5310d8b/',
  'domain': 'baovinhlong.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 852556,
  'url': 'http://gan.39.net/a/260621/x3wnanf.html',
  'domain': 'gan.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 570069,
  'url': 'https://tamanhhospital.vn/tin-tuc/cuu-song-be-so-sinh-bi-thoat-vi-hoanh-the-tui/',
  'domain': 'tamanhhospital.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 905348,
  'url': 'http://tnb.39.net/a/130607/4186274.html',
  'domain': 'tnb.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 909349,
  'url': 'http://wei.39.net/a/110821/1781775.html',
  'domain': 'wei.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 681131,
  'url': 'https://tuoitre.vn/loan-thi-truong-affiliate-ai-chiu-trach-nhiem-chat-luong-san-pham-20260130000926371.htm',
  'domain': 'tuoitre.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2475161,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201511/856792.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 722587,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/covid-hom-nay-12-5-them-2-439-ca-mac-moi-1-benh-nhan-tu-vong-727956',
  'domain': 'www.qdnd.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 732544,
  'url': 'https://www.vietnamplus.vn/to-chuc-y-te-the-gioi-dua-ra-khuyen-nghi-moi-ve-dich-cum-gia-cam-post942110.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 44335,
  'url': 'https://baogialai.com.vn/gia-lai-huong-ung-ngay-asean-phong-chong-benh-sot-xuat-huyet-post280882.html',
  'domain': 'baogialai.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 901143,
  'url': 'http://shen.39.net/a/140114/4326874.html',
  'domain': 'shen.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 812548,
  'url': 'http://ek.39.net/a/20101018/1506617.html',
  'domain': 'ek.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 678742,
  'url': 'https://tienphong.vn/weilaiya-white-truffle-2-nam-lien-tiep-la-san-pham-dau-goi-tot-nhat-nam-tai-elle-beauty-awards-2025-post1735881.tpo',
  'domain': 'tienphong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 660759,
  'url': 'https://thaythuocvietnam.vn/hen-phe-quan-va-nhung-dieu-khong-the-bo-qua/',
  'domain': 'thaythuocvietnam.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 84125,
  'url': 'https://baoquangninh.vn/tap-huan-cong-tac-giam-sat-phong-chong-dich-benh-do-vi-rut-nipah-niv-3394596.html',
  'domain': 'baoquangninh.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 1143,
  'url': 'https://bachmai.gov.vn/bai-viet/canh-bao-nguoi-lon-cung-dinh-thuy-dau?id=1e8dc2c0-88db-e42e-d5db-cb9137df465f',
  'domain': 'bachmai.gov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3160976,
  'url': 'https://www.baidu.com/bh/dict/ydxx_8424738550787796805?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 26481,
  'url': 'https://baochinhphu.vn/cac-dia-phuong-can-day-nhanh-tien-do-tiem-chung-vaccine-covid-19-10222060115443633.htm',
  'domain': 'baochinhphu.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 99635,
  'url': 'https://dantri.com.vn/suc-khoe/dung-thuc-pham-chuc-nang-nhu-the-nao-de-khong-loi-bat-cap-hai-ky-4-20251009161639294.htm',
  'domain': 'dantri.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 854825,
  'url': 'http://gk.39.net/a/090624/900447.html',
  'domain': 'gk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 702221,
  'url': 'https://vnexpress.net/bo-y-te-virus-nipah-co-tiem-nang-gay-dai-dich-5041215.html',
  'domain': 'vnexpress.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 64079,
  'url': 'https://baolangson.vn/thanh-tra-kiem-tra-kiem-nghiem-san-pham-duoc-tiep-nhan-xu-ly-cac-thong-tin-ve-thuoc-gia-5044678.html',
  'domain': 'baolangson.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 724965,
  'url': 'https://www.sggp.org.vn/ho-tro-con-dao-thuc-hien-phau-thuat-noi-soi-co-ke-hoach-post817576.html',
  'domain': 'www.sggp.org.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 853240,
  'url': 'http://gc.39.net/a/141221/4540391.html',
  'domain': 'gc.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 876479,
  'url': 'http://pf.39.net/a/2011418/1666222.html',
  'domain': 'pf.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 857643,
  'url': 'http://health.people.com.cn/n1/2026/0701/c14739-40751470.html',
  'domain': 'health.people.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 80524,
  'url': 'https://baophutho.vn/4-mon-pho-bien-tren-mam-co-cung-ram-thang-gieng-co-y-nghia-mang-lai-may-man-248768.htm',
  'domain': 'baophutho.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 118141,
  'url': 'https://hanoimoi.vn/dung-de-thuc-pham-ngay-tet-thanh-ganh-nang-cho-gan-ruot-734057.html',
  'domain': 'hanoimoi.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 98260,
  'url': 'https://benhviennhitrunguong.gov.vn/thuoc-diet-chuot-va-nguy-co-xuat-huyet-o-tre-em.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 90572,
  'url': 'https://baotayninh.vn/hanh-trinh-tan-hien-cua-nhung-thay-thuoc-uu-tu-140074.html',
  'domain': 'baotayninh.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 904470,
  'url': 'http://sj.39.net/a/140814/4447769.html',
  'domain': 'sj.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 91319,
  'url': 'https://baothanhhoa.vn/bac-sy-benh-vien-108-da-thuc-hien-thanh-cong-300-ca-ghep-gan-267964.htm',
  'domain': 'baothanhhoa.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 794284,
  'url': 'http://byby.39.net/a/230524/13823332.html',
  'domain': 'byby.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 98535,
  'url': 'https://benhvienvietduc.org/chuyen-gia-benh-vien-huu-nghi-viet-duc-tu-van-benh-ly-di-dang-dong-tinh-mach-nao.html',
  'domain': 'benhvienvietduc.org',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 720463,
  'url': 'https://www.pharmacity.vn/tang-sinh-mon-sau-khi-sinh-lieu-co-nen-khau-ngay-lap-tuc.htm',
  'domain': 'www.pharmacity.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3530740,
  'url': 'https://www.familydoctor.cn/a/201309/512842.html',
  'domain': 'www.familydoctor.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 900894,
  'url': 'http://qy.familydoctor.com.cn/a/201405/609715.html',
  'domain': 'qy.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2551069,
  'url': 'https://v.familydoctor.com.cn/a/201509/816823.html',
  'domain': 'v.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 766444,
  'url': 'http://ask.familydoctor.com.cn/zt/mn.html',
  'domain': 'ask.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 199047,
  'url': 'https://mega.vietnamplus.vn/thai-huong-nu-anh-hung-dam-me-lam-cho-dat-no-hoa-5415.html',
  'domain': 'mega.vietnamplus.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2131080,
  'url': 'http://yanglao.familydoctor.com.cn/zt/scsjtfz.html',
  'domain': 'yanglao.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 836542,
  'url': 'http://fk.99.com.cn/news/410997.html',
  'domain': 'fk.99.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 583,
  'url': 'http://special.vietnamplus.vn/benh_vien_nhan_ai',
  'domain': 'special.vietnamplus.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4030514,
  'url': 'https://ypk.familydoctor.com.cn/267067/',
  'domain': 'ypk.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0}]

SOURCE_PROFILE = {'rows': 4394718,
 'id_min': 583,
 'id_max': 4420561,
 'duplicate_ids': 0,
 'hosts': 97,
 'host_counts': {'special.vietnamplus.vn': 1,
                 'bachmai.gov.vn': 2783,
                 'baoangiang.com.vn': 10738,
                 'baocantho.com.vn': 12016,
                 'baochinhphu.vn': 2302,
                 'baodanang.vn': 14500,
                 'baogialai.com.vn': 3290,
                 'baohaiphong.vn': 16286,
                 'baolangson.vn': 1981,
                 'baonghean.vn': 15941,
                 'baophutho.vn': 1288,
                 'baoquangninh.vn': 2912,
                 'baoquangtri.vn': 5483,
                 'baotayninh.vn': 1198,
                 'baothanhhoa.vn': 782,
                 'baovinhlong.com.vn': 5113,
                 'benhviennhitrunguong.gov.vn': 1200,
                 'benhvienvietduc.org': 425,
                 'dantri.com.vn': 2063,
                 'giadinhonline.vn': 16729,
                 'hanoimoi.vn': 1283,
                 'hellobacsi.com': 11747,
                 'khoahocphothong.vn': 8143,
                 'laodong.vn': 31282,
                 'medlatec.vn': 24762,
                 'mega.vietnamplus.vn': 2,
                 'nhandan.vn': 5225,
                 'nhathuoclongchau.com.vn': 79598,
                 'phunusuckhoe.giadinhonline.vn': 27577,
                 'phuyen.baodaklak.vn': 6252,
                 'pmc-ecm-healthblog.beta.pharmacity.io': 5333,
                 'suckhoecongdongonline.vn': 154503,
                 'suckhoedoisong.vn': 85823,
                 'suckhoeviet.org.vn': 5627,
                 'tamanhhospital.vn': 4261,
                 'thanhnien.vn': 72111,
                 'thaythuocvietnam.vn': 2927,
                 'tiemchunglongchau.com.vn': 13281,
                 'tienphong.vn': 3021,
                 'tuoitre.vn': 3893,
                 'vietnamnet.vn': 15543,
                 'vnexpress.net': 2055,
                 'vov.vn': 9784,
                 'vov2.vov.vn': 6899,
                 'www.pharmacity.vn': 69,
                 'www.qdnd.vn': 3700,
                 'www.sggp.org.vn': 1853,
                 'www.vietnamplus.vn': 3695,
                 'www.vinmec.com': 26129,
                 'youmed.vn': 7189,
                 'ask.familydoctor.com.cn': 3,
                 'baby.39.net': 27621,
                 'byby.39.net': 439,
                 'cancer.39.net': 8415,
                 'care.39.net': 15134,
                 'ek.39.net': 3222,
                 'fitness.39.net': 12066,
                 'fk.39.net': 9812,
                 'fk.99.com.cn': 1,
                 'food.39.net': 11024,
                 'gan.39.net': 5027,
                 'gc.39.net': 1746,
                 'gk.39.net': 2059,
                 'health.people.com.cn': 1348,
                 'heart.39.net': 5965,
                 'man.39.net': 5320,
                 'nk.39.net': 6650,
                 'pf.39.net': 1538,
                 'qihuangzhishu.com': 23637,
                 'qy.familydoctor.com.cn': 40,
                 'shen.39.net': 3223,
                 'sj.39.net': 942,
                 'tnb.39.net': 4178,
                 'wei.39.net': 3894,
                 'woman.39.net': 11687,
                 'www.120ask.com': 918479,
                 'www.cnkang.com': 963438,
                 'www.wujue.com': 53341,
                 'www.zydcd.com': 78680,
                 'yanglao.familydoctor.com.cn': 2,
                 'article.iiyi.com': 5641,
                 'ask.39.net': 337375,
                 'baidianfeng.familydoctor.com.cn': 3798,
                 'bingli.iiyi.com': 24990,
                 'jb39.com': 7306,
                 'jbk.39.net': 19142,
                 'test.pmphai.com': 15120,
                 'v.familydoctor.com.cn': 10,
                 'www.a-hospital.com': 169339,
                 'www.baidu.com': 2631,
                 'www.familydoctor.cn': 44,
                 'www.familydoctor.com.cn': 447453,
                 'www.msdmanuals.cn': 6173,
                 'www.youlai.cn': 46119,
                 'ypk.familydoctor.com.cn': 1,
                 'zhongyibaodian.net': 146700,
                 'zysjonline.com': 243347},
 'first_1000_host_counts': {'special.vietnamplus.vn': 1, 'bachmai.gov.vn': 999},
 'sample_host_counts': {'article.iiyi.com': 1,
                        'ask.39.net': 1,
                        'ask.familydoctor.com.cn': 1,
                        'baby.39.net': 1,
                        'bachmai.gov.vn': 1,
                        'baidianfeng.familydoctor.com.cn': 1,
                        'baoangiang.com.vn': 1,
                        'baocantho.com.vn': 1,
                        'baochinhphu.vn': 1,
                        'baodanang.vn': 1,
                        'baogialai.com.vn': 1,
                        'baohaiphong.vn': 1,
                        'baolangson.vn': 1,
                        'baonghean.vn': 1,
                        'baophutho.vn': 1,
                        'baoquangninh.vn': 1,
                        'baoquangtri.vn': 1,
                        'baotayninh.vn': 1,
                        'baothanhhoa.vn': 1,
                        'baovinhlong.com.vn': 1,
                        'benhviennhitrunguong.gov.vn': 1,
                        'benhvienvietduc.org': 1,
                        'bingli.iiyi.com': 1,
                        'byby.39.net': 1,
                        'cancer.39.net': 1,
                        'care.39.net': 1,
                        'dantri.com.vn': 1,
                        'ek.39.net': 1,
                        'fitness.39.net': 1,
                        'fk.39.net': 1,
                        'fk.99.com.cn': 1,
                        'food.39.net': 1,
                        'gan.39.net': 1,
                        'gc.39.net': 1,
                        'giadinhonline.vn': 1,
                        'gk.39.net': 1,
                        'hanoimoi.vn': 1,
                        'health.people.com.cn': 1,
                        'heart.39.net': 1,
                        'hellobacsi.com': 1,
                        'jb39.com': 1,
                        'jbk.39.net': 1,
                        'khoahocphothong.vn': 1,
                        'laodong.vn': 1,
                        'man.39.net': 1,
                        'medlatec.vn': 1,
                        'mega.vietnamplus.vn': 1,
                        'nhandan.vn': 1,
                        'nhathuoclongchau.com.vn': 1,
                        'nk.39.net': 1,
                        'pf.39.net': 1,
                        'phunusuckhoe.giadinhonline.vn': 1,
                        'phuyen.baodaklak.vn': 1,
                        'pmc-ecm-healthblog.beta.pharmacity.io': 1,
                        'qihuangzhishu.com': 1,
                        'qy.familydoctor.com.cn': 1,
                        'shen.39.net': 1,
                        'sj.39.net': 1,
                        'special.vietnamplus.vn': 1,
                        'suckhoecongdongonline.vn': 1,
                        'suckhoedoisong.vn': 1,
                        'suckhoeviet.org.vn': 1,
                        'tamanhhospital.vn': 1,
                        'test.pmphai.com': 1,
                        'thanhnien.vn': 1,
                        'thaythuocvietnam.vn': 1,
                        'tiemchunglongchau.com.vn': 1,
                        'tienphong.vn': 1,
                        'tnb.39.net': 1,
                        'tuoitre.vn': 1,
                        'v.familydoctor.com.cn': 1,
                        'vietnamnet.vn': 1,
                        'vnexpress.net': 1,
                        'vov.vn': 1,
                        'vov2.vov.vn': 1,
                        'wei.39.net': 1,
                        'woman.39.net': 1,
                        'www.120ask.com': 1,
                        'www.a-hospital.com': 1,
                        'www.baidu.com': 1,
                        'www.cnkang.com': 1,
                        'www.familydoctor.cn': 1,
                        'www.familydoctor.com.cn': 1,
                        'www.msdmanuals.cn': 1,
                        'www.pharmacity.vn': 1,
                        'www.qdnd.vn': 1,
                        'www.sggp.org.vn': 1,
                        'www.vietnamplus.vn': 1,
                        'www.vinmec.com': 1,
                        'www.wujue.com': 1,
                        'www.youlai.cn': 1,
                        'www.zydcd.com': 1,
                        'yanglao.familydoctor.com.cn': 1,
                        'youmed.vn': 1,
                        'ypk.familydoctor.com.cn': 1,
                        'zhongyibaodian.net': 1,
                        'zysjonline.com': 1}}

BASELINE_DOMAINS = {'www.cnkang.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.120ask.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.familydoctor.com.cn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0},
 'ask.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'zysjonline.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.a-hospital.com': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'suckhoecongdongonline.vn': {'error_type': 'not_tested',
                              'requested_additional': 0,
                              'selected_additional': 0,
                              'shortfall': 0},
 'zhongyibaodian.net': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'suckhoedoisong.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'nhathuoclongchau.com.vn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0},
 'www.zydcd.com': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'thanhnien.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'www.wujue.com': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'www.youlai.cn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'laodong.vn': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'baby.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'phunusuckhoe.giadinhonline.vn': {'error_type': 'not_tested',
                                   'requested_additional': 0,
                                   'selected_additional': 0,
                                   'shortfall': 0},
 'www.vinmec.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'bingli.iiyi.com': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'medlatec.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'qihuangzhishu.com': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'jbk.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'giadinhonline.vn': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'baohaiphong.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'baonghean.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'vietnamnet.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'care.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'test.pmphai.com': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'baodanang.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'tiemchunglongchau.com.vn': {'error_type': 'not_tested',
                              'requested_additional': 0,
                              'selected_additional': 0,
                              'shortfall': 0},
 'fitness.39.net': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'baocantho.com.vn': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'hellobacsi.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'woman.39.net': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'food.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'baoangiang.com.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'fk.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'vov.vn': {'error_type': 'not_tested', 'requested_additional': 0, 'selected_additional': 0, 'shortfall': 0},
 'cancer.39.net': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'khoahocphothong.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'jb39.com': {'error_type': 'not_tested',
              'requested_additional': 0,
              'selected_additional': 0,
              'shortfall': 0},
 'youmed.vn': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'vov2.vov.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'nk.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'phuyen.baodaklak.vn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'www.msdmanuals.cn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'heart.39.net': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'article.iiyi.com': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'suckhoeviet.org.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'baoquangtri.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'pmc-ecm-healthblog.beta.pharmacity.io': {'error_type': 'not_tested',
                                           'requested_additional': 0,
                                           'selected_additional': 0,
                                           'shortfall': 0},
 'man.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'nhandan.vn': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'baovinhlong.com.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'gan.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'tamanhhospital.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'tnb.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'wei.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'tuoitre.vn': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'baidianfeng.familydoctor.com.cn': {'error_type': 'not_tested',
                                     'requested_additional': 0,
                                     'selected_additional': 0,
                                     'shortfall': 0},
 'www.qdnd.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'www.vietnamplus.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'baogialai.com.vn': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'shen.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'ek.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'tienphong.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'thaythuocvietnam.vn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'baoquangninh.vn': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'bachmai.gov.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.baidu.com': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'baochinhphu.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'dantri.com.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'gk.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'vnexpress.net': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'baolangson.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'www.sggp.org.vn': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'gc.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'pf.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'health.people.com.cn': {'error_type': 'not_tested',
                          'requested_additional': 0,
                          'selected_additional': 0,
                          'shortfall': 0},
 'baophutho.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'hanoimoi.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'benhviennhitrunguong.gov.vn': {'error_type': 'not_tested',
                                 'requested_additional': 0,
                                 'selected_additional': 0,
                                 'shortfall': 0},
 'baotayninh.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'sj.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'baothanhhoa.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'byby.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'benhvienvietduc.org': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'www.pharmacity.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'www.familydoctor.cn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'qy.familydoctor.com.cn': {'error_type': 'not_tested',
                            'requested_additional': 0,
                            'selected_additional': 0,
                            'shortfall': 0},
 'v.familydoctor.com.cn': {'error_type': 'not_tested',
                           'requested_additional': 0,
                           'selected_additional': 0,
                           'shortfall': 0},
 'ask.familydoctor.com.cn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0},
 'mega.vietnamplus.vn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'yanglao.familydoctor.com.cn': {'error_type': 'not_tested',
                                 'requested_additional': 0,
                                 'selected_additional': 0,
                                 'shortfall': 0},
 'fk.99.com.cn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'special.vietnamplus.vn': {'error_type': 'not_tested',
                            'requested_additional': 0,
                            'selected_additional': 0,
                            'shortfall': 0},
 'ypk.familydoctor.com.cn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0}}

EXPECTED_URL_COUNT = 97
EXPECTED_DOMAIN_COUNT = 97
BASELINE_ARCHIVE = ''
BASELINE_ARCHIVE_SHA256 = ''
SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
CRAWLER_SHA256 = '2c89f0c40efb4f3e17f8e4763f16b969ae810c4847cbc94e4c7ebaaf1ccfc999'
CRAWLER_PAYLOAD = 'c-qZ<ZIdKNdEoc?6~!vJ>R$EK^vv$c(ySU-tpsdHTFI*woMW~rYPzeZOVi!es;ZtB&CUl*U;~FP;J^gj*<lHeumcRjHpdw)cYIid{>MhJlK6Cg;hwk5%B;%j=~={`Wiwk{mHFh`lTSYHdEvr^FC@<->5DgRd4nuEoUEs_$+0(2CTS6PH&gHRMVuXblV~@d<Q;FC4(Z<_i;`I~+w;=QOJ>9P2%dur7cQ)gvUKW&;b>7TvN#OAWI9i?!i#3Jw1|o%o#kt5>Rq-skFq@GI)ijFi3e0dRl1%oW<{KJykR_w7L#I_3<_0vHdtg?JSzfjN!PhO&f;kJa+*$_JBkO3B8BQv9uCs!d=eM&P}eWwqhgZms(SKf8qK1;IO9^osECVX8mltuHw+*CzLCzP>+v*7Cc&aeCb}CmFf1PyX{hR{f$zPM%+>Qa%E!<R^^^aB7r`_xBB&VYFY{<Pgn{Pj#cO#wQ_s_=7;|6d@MH|@W&9w|Su`7_Q~gVKCm&<i^{;fW9~b%;E?QnB1Kn(XtRD-SYgl>x!WiUYmJ9${Xb9A^MK-~4(afms!MoSeWY)old_GAEt7w+)rUjMHRE1mk>a)otewkitT$&$8S(YBcQo{^XUIw`Miz23vgX%|s4=7&H$Jfirfp>XPjx>u#$s__ylfe{|q3Y6JT!f>^aBb}i&)s_E#!EN7p64Gt;jg`V4WC|2-$*8tXyZ!I^;%zqt&$$*-p$+IW;f_w_29)5m#=z9Ph4($*XDqsUyOI3Pl}Bzm;Nlc^n}-X{`0qAe4*n_lKt5GTs+uM+urptv>$J5UIGC4f8Hz6D9RG4;;-F$>6w>qhu1#$95hn4@2{<`!7lO^#h}&RUi09;EQaCCG*Sn%^suGy8jyL=4w5__r5WH>t1Z=s$sSBg3m4UfB4`}s<7n&36I>t|$462@)rU!pLoC}uJQWa!Niilw<(EM^k7q4E+x6SHC?hW9Xp4YJ1EY@TbKAbF-Ka>X$spW4F5;Y8>3GH%@CD3O*#-R;u>*<Bbbpv+E&h@BZf6Ui0YLNeB24$`Cs!%r=^Ox2O&nD?i>5Jw4DiW&+#C79iQYB=qE0ddn636{P)z532_`tq5}+R3W9n=9DU0WmXb?+Zd4yaXX@ZO|-F;1l=NLSe3x<p7Ja0ic7|Se2avkM^B<X!Nn&dD(B<r)Hx77~f8SdnkzbHoQPx@`u<Rp!TWs5dGPy<*XZy76#;|`I`wg@ClL9YrFD^8@Bus!qUG5CIfw4-H=C3B6W4b$%5nfV^zvWKS?!)+SXI0d``0%VV;r3e&#3a}8ZD1h_<<0k20oM~*E;ij1+Z^SJes{q1Bzzu^e24y+_=4Y?_9WEmv)gt?MGU$AVhG4x_?@Z`f9OR4LR_5>ADZD$2Za3=oAIE~8KyRCa(xvarPA+v$8K7>AXZI+`i!7P9+BAJfIDH5w;K`s4wA-qTjDHTjgsswo1759Px+v5R!?ywa=B^|ouZ15sJRi>9WCXi5z!inD(gw>FxuSkY37j2Yh$f5pIUwFy%YSK60BQH438L09k_IpG6rigx!UY~SG^vAen!=CBo5hFT`Z_ck$mHeeA{)e3|M-;<f)VcN*Mpb&q`=dsWqyYt%tt{!7y};=@}Yvbk$a!$c{_eG#Q6lQ@%z{dV1s}@1jX?@&Vj&&39uc&aI~rrMoBywwy<`a3NM|7=Z@yUI>m4#F*Gk30>?G<a9MyygT|4)T1;oT8geqsc~HD1qg_3wjw;%tzW3Ywj~Xdd3}%bT1UM>S&TvHYK|s$yih)s>4T=_&X$Qb`&5r?XU=k)k&+*-|A-tI6x$u+21>n{IjwQ7)H=fn~rk1el!W*{u^(e`J7B_r+%zO3L3tqG+#wifVV=u|Qd@&j&g9PXsq4hjV;V^;buX+ctA@NA`Mli@@Z#RbT@FOk;W2D&J8dTYh;0^8uSU8}(Sx7H{IF++UYi<l0tKm{WB`O?n(N-5$IEBfYBzR(#<z>bZV-lDfV8v#z^8rE!7q+di33l-m!NETM(zqz*`KPxxHg>+WK^wm$WafXh=^yzwW;Aj&<uGI0-U-zHM0P4uul@rb5_o?kW!XfLfemQtQhR5suPLEfgV$w;EB3v|d){VskojWTG8Zv8z|O(uZ+lMzuhi|<2VoYk!v>hVk%7E=pCb4*$S{Fr8E9r1e(L#&P>}Gjio(gqyysqDM3Z$q6o3*<(g6uN%-3WwZi_70UC=wk0W6pE*Ab8+JdGHyUSEKW1QUyLPJnFYetew6w#;B_OYhqqugk3-juY$$jEkyS6Q)OHBcH4s{6AHxVH&;FQ~7jt-H4*B+MfE)hN+h)gRR~Zi6pp1{|MoY<DojFfWbdVVU=-8N(r!q`$PzL9M9^sVA<|$S7&4Z3|L2POOVT2OSWdm-nX09$I3Q4|1QD7H1Cl)NVIgot4UEstx}U@n&3fe&3bj2ksKp9Tsf*xUp4$3L~~%ohpiK&|J!=1h^%dUM238BfxhCysXZdl6gV(~)CD*cAg71`@V`Ace2mez>vcv_t+B_d*E>|M5Ae^^Z^kCx+C5KpTr1<l0cRe$Mm_Z!_e++N=Ck9$hV4zF>FzM{vTZMu6O^GBQC1-3PEkYv&Oa)OjBp<MlMF-p4hV7+#4mv4Dv-|Acg;FcEj-caH6FGtYC*kOM$%pZrXd-@8J7YjT3=6w>kJL|x%>o3t@vm#S>(w<+}6y)$kXlO34|?`4dwBKnUN;`0*+ePl5j${ggN?D&+jlq7`S@*w9k{(?pxceY^J(eCl@bf=F$=)pQ&T`v|KCI!R6~^Y!+^cX19UTxenB4i5trRSBNb=F(921(MJ(d__p<RI)hF$_|w2Rb=A3W!eiU^x)pNHg!<I35G>}%$g~u7gwe^|^o_aT4z(4NzN)0lVwy1S420PB39cH$<Ws5?OI2aQf{<A*t^{>Xbl-iP8dyG{+)4!o4CgfD8N-b*rR4>-6CNyHQ$9yW@;N%PKbLK2786JVHr9ozbS%R&u9H4&>8&ss@Qoh~K+J?7gA&>$P41+lrl?YemWhF#Hcgi;6j>8BfV~CC6GjEnHB{^%`onzXF~0Gy-T2%hTigac?a_wP@#4+`Rp%(#!xDoun=f*FM<*1%BE_z6jF8t0J^>daTNpNVo=y&8IP^MtK}GwrZRmJJNHT%pGyorsEx}T$4dOdV3Y{OGA0u<cGp9+g*+E4+0h$mh>MT)PMK?oD#OPZ($9@B+0lzfYL9duH2o?w;SNrcAt7|J5v$G7TO$2~>s8D_qVZ&@OQ?$RNJRoNuB<UizBv)O&ZRpM571HCKzQz5Z!c@}$o1mo=qTi~44eSSC3v3p69$h$rs!S|TX{;AqLJPof$nd35Fav3E5!-Z>8iAHe-4Z})SD|w;?f_C{-5X@-)khY&)mQqpf+XbMB9+07XE#u2oF+NRR)OEp2UOvKt};3dQ7MHQOE^5Yx?S(-o>c+`GepyZwp$ljuS+hW1|Zq<u12Gq=-2A(rl9%gkyRlW#;EK@GeHY#@afA+v$m6~5<Mw`p!3aaPO!Z1X11DU4o84HqH*q+J03ufL64M018%*en<h&Ps1s1@;rGk;qhtnjfKT)KyzyMikq6+<DuP4>lmS$<UUoEz_Cgfhpb{_<RI^?RXbdba41_oT3QNT`&Z$+lgaoqHfCMr>M%sgm+!3f}jhLxGh;n5QX}Ll&i<*PxE=vc1K6I}1=v1i#t{+ndb--<p`Y*L|u6|)plzd@s=CZ7-2_$T_^H5XR#Y_$<=fHd!KA-peLul?Clw^geMm}je2%4A#=%KWYdd*XKHG<P-ENi&7Hkd?t?p;UoD({u&jt21@6#_ySQEM8)0hSbD2%^wrMEo!i3$%R}U|sBiO4wqtL0w1%kO8KL^5bHTvSpy*F$5%yT(a!ZXHl&cXN`S;(O?HA(Q$7x03Hd>C9ppLu3qYR7g0YCqC+0)w|G2t>cm2d!M>isq=sl?0>Hh^D+p3PvX~%W(6rSI2b9;@ak#q}0?zlI>KZ^sFiceb7sZ5NRA-EaLr*j&c~;$5rzkF=jJh1L4X843sHq?EWT<=<S&Z44%12YwK!VqD!EY<BC}EAOvk#X3%1xlFRi+D2SnZ-pe4ge>0o;*NAe%JXY>in*60R%25dR(B>NwAi@;~2=9JocDA6A8D;!_FCe7icN65UKl<=a&u8R%I8GG3lq%o&KL9gh?ZBC}Q&0rgtb+@UWvbj4I1v{t0-5p{9>a|;$7DOMU4ZCwmK3cx-wLV6%2=^OV5xSQ@uYUL9c#;Ha$hDJP+!nh_(lT`v#Yc*hkYf;^>ti`gv7_*NWDOd4&NxO}`MMNBHY1MrJJSbX~o`6z#KbN*=!1urgv3p_`;8(cIPCLf`oMQtOLXsU**;<Ry#OfElK_i`__Sj|!xF*y-i}z4feN1W*1FuL^U5>RvWt1tA>w+Xzh_1Fp7+jqeqxl@z3^)R*b)z|v9)O1=F{m62320QsuV8&yCXf=ODrFyz;1y}7eMTMd8;FLWm{JVWiT^5yo9oy1fIWi^`|365z_kG$+$1Q&DsGI6>7)aE3@H-;#KwF$y83#r8$8vyxN(u5uK1_zss}^8ZJw3Skh{$QM8t7A)bpfFQZ3t<!!AawVJ=P6<}`8%wkyVRfOss6pv+IkBEMZ)C8@M&+Ll+qD`k~eb6_71(;6}m0WkO+KVh>8vF)zDkq3?H8SxL^lW?G4zI>^D9?&ovsElN7h`SM?8qGaMzBR{Qkt*{RQlR(%9orO>558{rf-QX>t<457h6iX01TJ+Bw(vsCG>Sqa0Vdi^9#bgip@&v78>OwvTz0JCLe&-wu)adQo>~;<6`t(28_a>q7c5BTp**&*J3E*AJKa8d?u2`jbT^tb5ovQG7qdJX#R`t!sbYe$Wo1u=DIOn8HAtaON_oX9@-kjzFFEAr_{9)*7wCTCUPX&Ro58~I@)MpGJ>_A=rcWfqVGgp-te6}#hwX_`&902p4Y^}Qel?}j#2BDkIH`K5I5vCouvQQaUS9yd7_=mw$f8)^)zDD~*)7NJt<$6g(-dl?;0gi+@k~UARha{FgRhf7L3F>xE8+1m1!Chkf<qSQ!C?gFDp=PH4Keo2*bYFZ?Iy7b%#`%iJflc?f%8N<3rFXx*lA9o*5l!Bqc%R(TQ(D`;OhD2V)s~0!2nnwaJ`#VbP?2rd=kg=7Iv`mu{uxMjDyE&rSlFUm$ypE$fzgW|4(*6OXSwbIA*Xx1A~62<b-QYEA`M>fwWetj+=%1m}cIg@@be8P{hYAROAaYT6S>5+hPNoW`Zl#9eN4PsVLr}O&CmsLv6_l6mq7Bqp9d$(0+fYRX@3MAp--BngI8t4aSQZ5|si}`V&&M!j^HC9FZ>)loq2R0c9(-sJ>cHT)A@TiFWhqa3hbiXAQw>H<m@gl8S$7nd!87-BUOf(ISOLxe>UNf;UVtj%HxbLTIO6sd1zY2dYMYd1opcM`0}V@A4Je8;vxiR@I;<?emVSj?c~A=EYSCt0fb@i~=s}{hw?jdcu8cL~9tO(l!YT+)+wpjh|c~MBl|u$y&qKSq#YkKDZ8MryEdm#&X@W2`Nscwtz(s3gx!RbL#GmqSkUKnJOFl2hpV4xB{XWLyq$D)>9qtO83$d>|p9*j8sVhq_@9?vf%o)5$sN!=I#<CaUK^-Mx}m==q)mKGyIrzhK6{(ytOmCv$RXj_yy@=fX^_=(|Jqt1oU}H=Zyih$`N$D+o1E3$CrT6z)T#{&-U4UM-jx0%LW6pgUn0cabAN{uo`W{QN@y)YhSF{ZfIbZn7B9vS2r{|yBMsFtCveQKsnvcP7p(yfhHSvrY-h^!i3caO3#}md&$gn7QwG5S)p-*KB5!#23HD-qrzXa7$TBG8(AVv5_`Jz<BbZMa!e?Rfxey`;@HXGKjF@tZl7qZ3umANJN1=2ce}l_z4=6Gc>z-ZxUrLW?}^W~L3-y0?G&Vk3vZ~|s#RU}1x>#4=+^rAiZnt=WC46pgtF0JJPl~L=n@0JiN7dSH}e!!39!mBZUX2ti%v>|2o!UMOsuI-U`SAuAXx&cv`u^lIazIX+`8k(z0JCydA{>X&9hRT&;WY=EmmbwKkHngj5FK^r>&lEVnMi!l{80<TwC2D3a8%wnyY&|l?CaSy8E@JChUo|20KsZEx#Ot<(_3|A7uQ5{NSLDkyEkYATVa4?gM*s2Q9zo;(#Z%t)BoM1mb@J+pm^G;FRVP5%_EYhCNW{B~mmon$SMMK9Nl#t(7>9&J+AZIL-Y8M~yi-U(KnY+C!%eO{i7T+orF$ENb3;pzOwb<XU;A40h%@*u?hmE-{rIdi0y!t;^m;`loZ|8Fl7=N+@xtz736uW^x-FlmVD(&og*4#gaAi8@K>!I~)EBJvC{jp}uEvP1pI$I-c&xHu>!pr%Lm-0<L$mzwI5cJp`;Fh_tF9F@GSX+E@Ox@cmJjq^0~0)(h#h^5gau#a@pT8;DyJp4z7?4hykL+$nmd$3=Sy+kdF+kwW}8;1wuqH`HOs<fA<H&@ab7QK!>s7@f%GJPorG*p8>c-Wz^ndscjvGM1uG1Ns+`XNmSJVQ-BoVrj42J_CPDZwNa6GxAG8k328{9W}9x!YXPrIm+MgZhJLk<+BIdf;zwtT<LcE9f1kJLNb2$$lQtrz*3HC&M<5*m%`lAd=vXunFiq6=IDQrgSb2cAA~+p?FA4zhd@jO4t>R!QcX47%C^*gGCfVRteq+ys<_##ofnS<jPZ(`b!4@BwbfrmT)Wf-4&_pJ3;$fgKYxaQo@B2oi^9C1Hs=ySp*$WX%(x`t0&Ay)(_bR}yFAsJS&e4T_Xc_1=Iv0JS4@)}{p{6`_!|$M3=zzt_?*dPHh7mAk%G8x<SkZ)KoXs!@It?FvHYje<mIa{K;X5(^;O3+Q0(H<p8%gK!m5T+%l3-m2MnGHcZxRlQ0wPWmeeK}BdR?#fddM7sq)q!nWo(kRe&rxbfOY_-qK9hvU(iv-YR1z@f^<^>7JedtUyD@2M2yt^c>yGU^Lt8wzvOuL)dX_k$OL117B{`w=}Nrh#e$P7}TkMzO4f=6VXo!Fa7!3w_o-&wDx)Sr@{DJ0|SjcXp;!VuT=Kh6W#9r%i)&Dy!v=q#bOSQCCj>O(;(kCMVoWWLJh*%s;9k@is7Sjf3qE4;6ullvbMO8H;K<~uNeZ;(tNb2R(0<{Q}qf%Z=a1qt%}PH!PcNPO9D;`ue`uO&pwFpFl;r#tai7(B1ep*+w@XaR(Y3<wzIz3Ka+x0aK6rNxv9yr)g-fhx%Ox;uWQ2_NKbtS)c2E9nXvp|YEW9NodH2}!zG96GW3_-HPku1o>Pmny~0kb7}PaCw9Ke3h3w_tjUXAx2m|A{yo(pTmg?I&Xp%skB!Qif>;MyppjA28y3%ekIt#Er()9dD7G@K>aUnrJ3hNWg;$9aG@rD{M^I*AQrOfIIF8^HmD0^3_e&ApnvI4%Y$pwu@?5d!kv8kt@KrFHalU}1xBC1<#zFArrM4+b%$4McE)2io_)^AGJL<4Aem=H(8#Yzto)9tCz<4F|Nc)Xs>#i!;JV&UP(EG|HtHx63mC9y<_R2q|Q?-AQ!uckUzn)gr;+%c2V_*!k^hNZgQ_W5)h5TAB<0@Aio>TP!7oQ>Kp+QKb1*ew|54g>0!ZZxIkKnA|%<{4`%?DbrLT_-UYXAdoFbCSlqAsVGZ*wh3x(6zbtZE}l9<PE9~FLe*lKZa5@J8>Gb?XUxxj;C%)qB_WX5X!rh4cPfh=V6E)D2AM#nqfDOXGCup$|N~(F-WA<q{RX=ljMIXQK-H4@gSIC3!+2<C4`b}@vi8UTew0P^?RJgMvqdaVn%!FajlBzXE=We+oCBsDL)iL9y1qO^_&%{Z4k{uQU|t-lwhumAkA`Bl9M|O$yOGrWg}!yH-&XWdTBOBd#!6kpwk|eI!e2aZ>>t^aHtBB0(?+aDx8k$3+`93ky2?{H&S96m1{?m)X^)Xk0PEt!^;t))l5edOTkpW9dqJ`O0Z>ET3nv2EP&HQp@yP-!As;;r=ycG6IrTU&1Rm*NJC&bS=-2qF;=GzGGFsPc6LXhx`n@DhHWD!i)Xdz1j`3Dzfq@lgCYx8V@J`HzH5^;t8eR-`EjjkDr>Z%E-{{--+H#pP^P-Ns)el4;PptxE-`f}ix;D1du@rP+Y}=E_3*T+H^>Td_EPP?(yu$Ds!LU=%D~8~Fx4zYHIqrePw;`?4;4MO<vDO(q~xGSgS95hpK$FYLP-aj9HJUj&{g#tYpKxZsf&NM$omQZTxRK64g2X;M*W!gLJB-GTf4No3^1eM1E>TgbgGXaJuFzpDBhEe){9xN3Y+%m@~1iebFgCIWcbASyk5}-iuCUwB5C1}9PYGZn3f7DX)ep5A_=ztgiBu+Dv44ov)DRJL#3xHom(H9Rrim#<T^~Ua9Zt=X*SA>bx1{=j9Bkm+G^XUHCg9lw=e0i??*E$(L9&Qm5@kS=R=XgPi_NE%}hwQWf+c=xa2~((yEyji21SH*<8M+;?+T0z}IPD)2XEKoWPMiQz<SX|MT;J)GEk^EKNAGyA+DCi~KWWs*O4K8HxnpMhrTIWH5=>A+gvr0@5*#7a1iSrfkxJNf1p|GE@YOqT_rDV;Uw0gaQf0G;Pw~E&6+j{$8fPSLp8(__q)n{y<sx2_hzAiT%@({W}0nWC8lyp~><+=1q>Km`!nt<IW*U?EvFvfpMCUnhp^3s`nBm7mRV<hY7Y385NLrB9kuT=46^{&FTKI{Q#hH&OcY|T^ZLA?aA?{Dz$5g#?Y?|iqY=M2avS0pt&fHC$zzmQ3gUWKFw0&^8xKTROR{0yHJ9cRf6qK-2!FWZsu3(;=-z(Q06k#6M=((=~-4w7izEB{`CnaXr2)eyFM=~6{wL4dK+N0ydxIHYPT%up4)=R5@rNhI%e;#=eHYY@9gPXD@UZZqM6{;NH8m2*9=4_y9K@G7&MvTe6g!S7|GkW1tAe}xvOuXtrUIM6s-~L^>a?1AcU_2sa26vKnOW=Kj40fIf^jn4(@!tsRaOTV{9ronv<)`#>T~+&m_a%1x5M6OQg8aFY}IJz=Ntw2}?F>>7h|0v1a>3sTvR)r&<bhQw7*$<8>NGwI$^vbw)#CQujTXWOWQcYZ%r+lv2cWo*GKj3y1L#$OfeCyt#M(tz+-K@4WZb+1{P};x_!Rb!T|8+1WbXF#f%K+HQUN6L*GfE0?cyl>ndFQByRi=Klu5F()@}0=0uKGyW%s8=stReDW_K2`!<8ofG^FuXM9@8{K3*LOgxyXUMI13T&iDVeZ+KVi*XCFdrS95Qhn}R=Fg=G#n-&@iXaY@R`V}LN-bHKR?!q{Na?s-zF+j5EZ|+_tJ>sV}}%hL^<fB0<G;Z&Ihm^qZuYSss}NzNXTm^G<hh+EcO>Iu1kCgec+f~OcKqKciYry@O}eq#I3W=xj(`XGML2cnDA^5rf^+$7ddE1Q+3KKWLs@@I>~)n0<&I4@lDr6K<flTZ#CW!8{^A6*fOv`m?n1y-DzF$E_6yVwVuy0x%FoVHYfn$EBz&~@Z7I3)L`b>Z)`1k#KwpoU}HO<f!Os?was#dF|9(2x*)RE?6{?A0Cm~i{whZ{@aL4Dg(mm*jho&JH|~G)ruXNsUc2d^!+8<0lBWgHlBV6(oN+hp<)L{RW{?jJi7xa(@iY5?`8iQ|N)^*Gxf5s#m4LLXpa5^bc3_?}Qd}kZPY0B0heI#bW~U>?R6nY7rQGLIa^uhc!!=LVj`S499)}(znaN<TxXDHEw4;PkIKcOGVsCDBqkJFT9P#8nJUk4HH^zP?B?aA5(r|P^2ErUfr3-z(rkT}s?PXa8^3ia37ZTv_W7W#0iDvmBABi7ZJF^_ChK|PMWYVLno=&b4%A5|Q$V$-AKrp<$G8`&_W7^ThiEFngw~V~sSXWRzW1QDDqyVyJ@X&8s(X&TIWMv7`z=C~#(?S}7a>Bk~+a@<hY5JMa?5Hk4$i3NRaL}lW=RWm<^WO4Iakp?*)q3$8qEDORKjS+fN%A&Od_gh{-YVhi5pMeMfP^r~%cZ&(g4~;_)LH7KtBX=k#-ie8i19Oqx1=Vyq^j01VKt2k8K%bdZ8r}PyUyvrQUbW+KPEdIEDM3&XgK1)-VTr)u})?Nvwct_n`Fj#V~`mbMAD2dz;fJ%{#|fvH?FwqUUM)%48YoFkHi9<pABkMRy;C0q|ziSE(SA>W~s@)crrAGhu1C8!Tj3(yur^DBa|%$Y0j*XO?v{4^+82PtL1<A-~R34*Z%c`Z~o*DU-|C;`R#wjm6uoF|LYHb_7=Wsw_WaJ)dPD)YBJuW?-qK9kfOUOGg3RFToQL3Z>qEj|LOe6$O~@~EjH9>jJ4zSZa>$amL28tgjcO3Q@9$8iYuc|i8y&SpmeX%q_E!O?}p%~IzOk+Dr0v6PEPmywvdv#NM+~HG?lj|uLW~Y4&Me=?)sm6`Pt8|%jT)%Fyb9z0IU0M>ua>8pl}}9UNiENZ*r3K3fw_VT1lE^A6p^I4m{ZSHpgkkBe^{A@D<&bL`a1hzu8E@6dk`(687P!@50;@jhEOdoq(4Xh`$I0>3GY}TIyN5LG5j$tV{ud#9G!l#$ZsN4O;)>a4u-2j;c9lhvlKsxTw57Oni317ynGxDzVm8`RYqST!5YJEz4hD$|;Y1@2m>jYfIAzfoMa?*h4G0#70YXXU5AJT2n}*fG~tfgtjEw!ZF<2qpzI<)hURutg6L~vY5^Q^sVsE#QWScTbDMsc=Z8~t{-P`(-|m^*F?*XpyW=ZILhdP6nP5@mLB|X=gx&Ye*e?0ojZr?{m0vWb^k!k8(LBzmnp8kZQA-Dp%tFk%_lBBdAYA+FzCiA(!uTl^2L7JY+qfM1q})CU=MU1bG(B>TTDi}J49ABy+-@Hs!u$~-Y!lYwzaprkN+I4<AbrfMte|Z^U3a|E}w>oAaIDz7b=EC)a5t^aEr+#9Zl(0ZZy8#oPnqYR!xhlOG@etoOX;8O<9m%<tw$Gr5ch<;Dt2ZU(8*p1ZsWNPQTjFVBGc>DEeCVkmzcU970mMeJW&ySkIS~v~=^EqB5LT=IjH2l9vyWPOzb%yoHV5(-jITNiakKsRZ_>pCxBiHm+V1Z7=HsXmErXk}8mUPlgmlA4{?auq*=EcG1SN5Pqo=Mw=iSlS)JpHO{#XwAYYO&B!My-PdG(k~h^>6!i!du0S0FZS3_%c-9L|Rkdnh1$98jTbYk1wFK!{)P<jbNCsm)kksrAN)s$iR{`-=UIrSrD=t}yR=hY_aQSAfsVg{~O6^FAKIgWE*7O@YOFVs37H6q!rbeqe6)IpdO1|3@JS!&?^suJ(^meq?gbYZmJr4HLgWzE1uR6!bA*1T2c_!DY-LsGrJGI<SQWx5GKre^KH>>A}M`vaZjg_<?wQU<Ltgu7K&_P@mD4&<fHiiZQMVa$91|$=-{O4@Q7A$);!Lk%0oeF-3K#Bj!{YcFMi5?`LLIxLRSfAU*z<E9@uJJ7N@I5Ibl|vaILM?o0GQ{*jB8Iro<I*f;VX1_c+o+H2^eWdrc05i0dJ1=0blLKk9SAj<riZhS9RQF@6v6%(U%Btgpt>X<xT6kGB&Z?>+tdRWg7#PFIEzU=EhH&4vqdEJmIWkZ8C}B9$3;>^9^fiUTkT5qzim+<>qZIEtXQwjiwQ_Qpj{$eB?7u_md+TQ=?%FvqE^Q&ty8y34U-R-;&Push{G{_1WOMNOo|^L`V}LAUAgqyL&FcHy2Tv$jw?D`bubu;+K`zwuUIJWK<$Wu1$X{H#kqF83co?_MQiFRhP?FtK$4*)eCG6Dqcoo&Jo$GPmo9g^J_ctF@DIFrYBU=0)vp5;j;u`N6zV(`*BLY=kc@^<pw5tTEuc>^xC}{{1Bk8#YtQLA!2sQ(=i^B30W0CxD`Lzqjv=>hQ_bz$R3$Q4-#VqrQs$j4ab<*V!>vCEY!m$3s9cO!;>}PnEgQxU>LtRX4iIVUiv0Boxp#PD%xml6o$psc7E5xKwvK#SlHH7KT@zqz7AR)XQGs4pS)g_ZPQNi;@URQEN{rDpVKLqqhu9hm5)|uK3Qhjbdtbf(qxZgZ|EKqV_5R!M|MI=Ry#JT)ef$15?|<+8x88rJuPokfzz2y1*rX6<F<?G{KtS}K6$c8rCzB~$#Y4mtr**4KGo~vVJ?9y`B;mfOSacU-eV`z8$pteTQ`42NRf9WqRksSAd>aWR&a7!E(N(=oXHi3krV3u}TAoj6l*4;4@M6|6vRP=S6LbOJUC5P;{+r|wil?zItSGt$Rh#m1LT8}*us$1H9ufjg*Qf^}%N?eOPUNGh(v&Zm4Tb+*(`2wD1%STY?q(AJksb0yju*F;5K8gImNw`-5Yp7uq-iX)se4(|rm>TFU~d}xqGTMzX!g~zX*_IkU<$|=0u}-TDZs&QAL&=ROD?_SAcgvue44`j@_fljhwnJUM8e{v2Oi%s#mbphSKc}T55k=N4}SfPhd=w)!{7Yw;dj3I@E1S+{cqnw8vgKWzyAHb?|<;kzxn8$Z+!Uezxwcd|4eVcb??LP{r$tg{l^dA{oTW_z4_6ve)I6Z{NqRO{0P5&@Pq&S!4Llaqc`7q_}#CNp8S7(=fj`<@ZmeZ{pjmIt+P%3?|<{lhi`uW;Sc}u;V*tbp#S@u55NBF5C7!{AN}fA5C7_G4}b9+eEZ(F(Y3~#jUT-G^Eo)$Jh=B&Z}<K`%^=bD0$!05;PvtSpTmp%{@bt^yAR&|aq3Oso3Y5ajyz<wShRNKE13`4JC~Z4Z8rrKXhsWC$0K#ZJnaRSuQWlx1hrfcgJ+V|LS#A|Y#MCW<}x7K(3#*sVN)rIl%NR^Xg9+d`aOr~AhS`=+7WKPtPyNe`FcOkqN_yakR~jpij5rsK(Ym-!Wn%a2F>F0$wX1%{M#!LU%1TWN_RQwGes<$tE&PRyG2FZw_#ElBBvn+WB|+PURG1h`C>8vfh!#qH}D!i2!?=b$S0t>%<fL}be()k>8aqyr$2FL2ii(Tcb}#gpSt@ri0q$|qg}L}O=C{Ur4~sYZnDK3HbA71f<yA>=tZB=#Si|MTA$vAlqh#06Ukj<W8tbJ&Jl9=HN1%gZir8j*x$t}Gg2+fpoe$=@b>@s!{7br_y6|h!=L{ED3tvE;BSBW!8iW*!>|AHZh=ui-v0RhuaL4nxc6P41P|VQJGrY~-T$X45Y-3o{^Z!BvTeUUcUF&uAy(X;Oa$8*0TEFD&LMEd)Pt8L-rRbeF36#Hynt)5?ox(8B-HR4u0Y1S1z|1X8964u79B)<N)Rvz=GX_J{O%uq`mK-N`O1fHedX>TnnSo-WG_eKdhiI~|DnnRj{QA#oa$StO8G&npoo)~j2t2<R1A-#gK)sc%|+F>*X(E}mO5VJ%evMqn@>eR+1Bw-6<KMUA)`(YHlQ}t%qnU|#pyANx2+k(GHVi|lu+btlczxEbdy|fD>7ec!J^D2Af)M{el4$_lmJ>fjU{|m@;J4@RKyY($7O;Y022K`9yd;HnvvdGR#soQh_~lr8DGNu?fj?JtfqQSeMAT3bXbqNG&7^6^_SQ)=~BS4mgP;2^~>WhlNjAA*t0Gzw7!$DtkdR6zC2YCOj&)5Jl23P)EIK&8SX@85<H#%j-alZ1g~=Cp1wt~r&={qq^SoZ8y<h94)jFlc3cneweFX!2HZJPf7NYuRK*NibHXxO{@5;TTyjz4Di<_9=G{^OODt%>-POQ+p?}TB7>GI<1CcB7h(wC6mVyM84;P9o6X1)3wbgEWRKqb99WoJ$=G202YY~g~*d#2gOl2dn+&X8W<y+mH7s&|~46*87>ClQ4??J3>Jqyq2ws!(APJPMJ)?TYa#(TR&9o`d?uyWcEoy)N4hU4WI%i4rM-!_G>Od>~j-wI8eq<bMJ&#m3Fu0Q!0_|?^H4NZ&wSiNtr1ZZ8dSHEDdEO_?SdVK#yd2g@2&_dmCY2SkDx(!-)|BP#hQDdoU?o{qD?k~L|p6^&EY6*uUoWw)beMu3J&XwaT-t@LznQyuQOs)fKm$%x<8$8sld%vp}3fjr9>sJKw4U!F4{@IsEHr(dN*GIZ9=tBUfDmamkH#x1)z|BuB%F|2hr!I`dXMOvjho`DJQL<!1WIGGGw#}#eHB!t<{)Ojmy>jEFn|QvmJny(?It=MKq`SVcNp*!aw_kT&GUl_=Hy`iy3ZwFRvo-h4g}R89N8;i|b#`0V5;}m)k3ou>h2=Q&<2;aG%G$)?CMXqClo^*%KV2|ECKR`sllivT{Xkz?diO{J=uM*?X<8*CgRLD$faZrB5@;Kz#d)WF%IhwCg={Y7s^k_%+K6lVQZqQs)s@%8XO}w|ztK)qi@$8+(MbX^407M5?7gcVUa%a&`F4mQ)*c^N7!w0Wp*YE7L<~gF%+T3V?Nx*3GB2aLptkZMDW?s`0ZKN_F$Q#%2~3@nvhW4dY>X3iR8MiCmQtIQ*TO88+I-^{a_oiaSOc;QUAk7z`-(S^lOK$#-EaYU9v#H?HRENKdH^vNZn$h*zCGey<JZekYLc+z%5iGUdE+=9R<c6UBO~$dl1tc^b_sJ(i^kOKOx*-0%c?JQP7ejKI1Ah@`Umb}0hAbnZ3WyKq=r7gh%$lN6BJ5^MyfB3F7mXqZ5Ct{Qh~!g?>Z?iRkSNPBHpIkCtrp@o#&3?!2<S53oB#R28>B3X!3)75uv0Z@-8TSI-)Cn2=c}56k|t~RE61Ds$!Li($Z@?4-7>8#YY{1O$iofi)l<*a3Un4tTZ3Cq;a70)$}3=y(a(y_UozMZOL=8L)fVJ);?YswOMnGw#pz18_cc8V_}r5$kRm2N+G|=SGrc6g>IL#v*R9aO74MTCTuCq%=&Z>hu((9Ego8WT*2YClwqwIkb5!tHTT?<iB?KvUspXePU5-1DSbu5oOXb!t_F~2(g4IS+OlBfY{0#<aA^SYMC6_*&Sv)Y=G8>1h~M5PO)ea6n)4Lq<1cb@6$-Et8t7dSqKOi`9}ZN0g>AK;d`urSTuk(x?WzH-tx!I;xHC>2V*o`9@DRwklI}UU3E~llJ>dlnnn%ab>CceJdsF~Z4NOQzVh9|_loPO3Dpw#h?WI@v9!i+Q0}ujoR6+qqP4AGA<IH{uCXl3idce6AsMItf*H^f*l&&1WBb}6(@C?Q=8o7>pGj$$BfZ(L$gj|%j&@YK0aVO-G?MILAvhy`C7DLlshAt2E3WDk(HkmZ*6>WliC#C|t$g5Xk$!D5fgDlDGJdXEUT}z{3a(bl5hfu>yTNSSZaqODr2ZPGxhR$?HT#hDye7&Skm|!MF=~ceJ@O*glnHR1<k2KbmMN1M8HRd1U<MS3AJ0Nz*0eVeT5(8%8Jj$3umUY08tDs1s!<w(Wi#!&e$vbn3{8~L{Q2n@-xfl&*#VYr`U@=$@L`|NDj=X!}rR&d!&;3WpL16s8`OLrDnWzo#IEg32<#%8;pxt>;?|k_5>P;s;Inyhiu{5vVRB0Np1bWq1H666AHQ~`D%3!0*8Um`Matg@DricJ4g)`zRE|w@MVOC;!q9T4AF&lOCu?ZgkzjP#rxh?DHE2F8Boz&JT<<O<hO=8ZxfwRx7C79RHJ}ZTp5<Z>%h9tvZRu*M$u_58cvm0=a)iNO_s|qmN&RcL<#ZOB2@`llt(#qTN4dDW7?IDVe#4s*UU%#b;7+tuYX4zs+q7Nt*WS&kh)CR$BYb3U^J(s(iDSx(_MMgShns8Q{R*Xqw+-jOqL^3UT@h8QUBkpA^kSGb;Sb-WpZ#n(-*9@=GW>BVD+!7tQJ_V`d6kSRxKT5lb5u{~m4T7iZS)52_1{c_~#ejFJQ>y(QW$9!-CP~EMqdO-RE7};<Xz2iMOd{@(qL*6mLtIixS3+)j+@P7OjQnng=<VTc_Rky+bV$c+gsMeKSIBJ&d>n9fP{h}_YZJnjR3R=z#S~=&@Cyne5#rV<JJhb1hJ8C<01C;<>pJR%qY9u7$oAkA6zkIGK{QKn>^tD=nJPyhYsiInk3;mpwP`CmjzG4EbiSCd*#{{tPp_OjVBms!Y~w{QSy@igpEhqsb#8%Cou5n59<83b5jj4rwsxvDWDsR0I03({N8*=5!PrIpgR+-R*yfFfYOfTKMEJ?_C4!)_OF8JUc^4bguoPnHE6e~_p)jk$Hl#NqA4<(*_SyucQUD&t$=;Yr7xw9B$DGbSBs7ILB^CnO!EZgS{gMGu$ObrIl|V^LuQ+0f&^+{5D8_lYwV|ViXF=aVCfXHE)`_{k<l{>S)po5kG>q?+9o{N09>QFv<~v7vjjf|2Su|x;t}8vnB?$~>_O@`qG*JtiE{E8$u!Pb%C<0RlBkmz*W0Mp-vtQarWEGH7^h}`oPHfuYE1@R>azLtTcq~Yx3NgZ8Bl^{-1#-1vIk42b%dl)O7gVi@uXNjmwoU|qPhok~@wui}T!?tRn!*K22Od_iUn2_aICD4Dpr|b$THpYRrUi0t(>f?sl-ewm!wHlg%jrP7jIOPBfX;35<AVUuVG_;6?TBKTL2*s*T0$Wd7eUlwHK6mSuJD{-ogKQ~VGt*iR`ApnAZX<DFR*d%Z}wMg0R$hwp-}HSMNO`j+9+WM(5!^&Dl{@BIpww_Cr6$es47pV4aR<jt1N~F5-}^3Px&q1iUIs^F-XvHjXMP%=P?9cL!d$o_{Sm_q|u^49B&S~CQppF`Bh0mh?$6~efEx%99b%D6~WXpmo<K2Io7Oec<6&|*=s-}(}tD8m8*6p_>9aMHN%KyA>iZxOy;GULk@r?D!t$%Ffht=rolMG6YM&MxQ^2VvxMuSt|YfOcFo&B?q4P_`{D+`*4TuyWTK3TMtCLBmHuU&nkwDQ?`sOw++Iab2H%ti3oyA91%>`WJ@zY5Cn=~2^015d+;6Ebpo*Q5{=ojw$fMxC27PS^mYm%fx2f1-hXRoCC(h(2pcMFb`Gz3J+%FgF$Y3g=L;hpl^AFzr9j2TD=XBQWNXF?eB+n%2i#Kjj)D)<YF06ak#}DqkIRhEiz|2#;V)`m<nfq^%)9OzO`A%qjdes};{}-Ie2lxICJ}4WK<}5%1aiKB=*b-em2V8_%MROK?fLdhte-1A7M+>9H@(95K5AOXlIpzOwhNf;PHfL9T)qhR@-95Pe;N5?ic}IW<6ZFJ2J0cVfyD<a!i8ow88@mtg{U=b6)C<eO-!RhK5AOX>XyV>q+nodgf+`P)UHUOi9!<Lj+HkP-w}Y)ux|_K<zW=XNoVDLb9!oU~a(Kd64j#PwGZ0CW2lw9Lh4co96E~jq_8#2(&Y%X?C#XN3m70Oc{kO_g%dqm}_kS6IZWb=mK;PxxdHVqStJn*C`{3R`0B(SHbKpJCUe0qU^y37^%*DtY0-}uX7|zQ8`u(@5&i%LN<qTDnZH!@r3^K_Oeett}^0Wr@fYkz)1-@X6GoZ`>yiB312r>l+2t;ZTWiNE<IfeYY-WTIt%ENURSMqNFC@$Mw??r0ouJ`$JUtkdvYvK1bHFI}ym)-T&*VjGzyS@A$zZ6hSF2aaV!ws$o<><m)@8qZ?)TbW*HYFGJuEZtvrj!lzD?=fpUX?Nryt!ad8O(Koo!Fr;)5jizIe<y0q>K08PK3DlU;gr!<<vGzbQ;}lmbl*`^W3cHZMC_o-*KjU@~FM*KzVp?!JW77|MJ1TAJ6tyYqk>o>TXtPgX&(sK4jl~;l}O12yZ?2@=LdF2U9wibig?WQr2{O4J-}@W{i*twsquijfxL`RAlL|cvp*dT$OjZeYw@{bxw|KisBSit!;Zk)s;kznjn|H$EBlja(CcErOX3WyKe1909jA#MgY<lDgLr*k0$G_ZOYYV4z`(^mJTqPUe98>Qiy!0#6^rPP3#gx_nl{$G?%i&kZVVIVGdtPhBj@9g}d>rw!`w9PR*Pmiwjm7=%~ZW0%L?{hbG^pV^@Y?6{8~qkxVYyA57_pofaJF`YPU}WzmnzSi%7FW}6Al#dIG=0<LNqCBi8!^pqnFtn!k+-t1DR!Nz37i;NdVG6^unA$~8%h7da<3E;Cmb&7%H-rzEllQ!2z@0)aTfESl_N|dwwr1jg-^6Bh3JUW{QN|lbN-+IA}b&?B>Ts~jM1W+=;TBVL>U(=?P=X3)YYNbS}zFpdhw2;v<tarH)-QI{oD;w~X>n-JZ<ID*n7MUXuqggxHo20uf|Dr5^Em4AP8AF#$)8HUuWi7n8{gWAi%r|CY23LTE!yE<^ngi_+gb!TV7nYmD&<C&Q>4B;~l-4i%R#pzGsanmTv~43>b6eXe+Qb~{avPe%c~Hz}(qC-1Vy)75r-&hM`FOZ=z<bX@7SOPf;H6dj4qv3|h9}a_5Ni`)Mbs=)fNrmT-PXqzGE>jK;S8T?F6IuUaV7Ow664}9(*C<oj?y&D^CwGjjQ&xQr<~OxS2C4`i_?|_8#QCXnSn<g&+baEl#1&SQTB;Py{1!d&2bHA-MfL@(fIJSLwuIG;VBP1<nd5MQU@G|i_=vfQtw)}NBr%}^reMait+pfey<mj0X9@tmt8d7QK=vrLPXPImw)A03DWvuO1_9Lv{zBW2DO=!d1;qway5vIhcQH%iin-lw@A9v^4HgyJo-8iriWSYq7AxAAQUDX06?Td(Fg;BT-+V1HwP)HFsv8Tt|l0cs+z@kuf|F--K<rwq_mawDhb}u0n(3EZ5&VLJ^uz~G0ZUv3R3;&Mae$<OM#_x2P_DO6IdFGM|Udl4O5K}vly(1>%hSI3eqinBY}c=N6msl#Yc+?9lIQJy)I<X1JLNpCKuqKB&b`03_PJsPY!<Dvq4WVpczF-efM!Ju6VDz*SB%0IC4npfCE}&)(RStLdO!hTD#OZ@x)Fpugr>3UY2M19lG{{`ygDVXEc098xkY7uEM|?MtG%phyn=Nr8A&leAt;#@RC4@HgiLH>TD=t*;vQ0wDWfb;vLpOMl@!<8Es8J!)DxN3jHX^TW^tJEp3aBm@|l}?CX`b%y~P=9S~_Kqhw@qT+lxVJ8x-WtP*ir?$1q1>w!E_rHI;$RG{l9VC)<QVmJ<RM`Gr`rLhftH?0aoRY9m%<+kJ*?vuR;U-{+00vxgS7KO2>7-3>*!jKZPgu>FG?`&QSQC`R68{^afN$bMgn#%RqYOnoYCQv+e'

PROBE_KIND = 'all_hosts_public_pages'
SAMPLING_METHOD = 'equal_host_reservoir'
PRIORITY_SHA256 = '608a80b2281daf534180ade41eaed2e458e6e949e51402eecdb4ace9ed5e66e6'
PRIORITY_PAYLOAD = 'c-qZ9O_STka(Dg;M6Xf+Ws>_L>qVwSr}S2Gta^HqN?O}5wcbL3B)Ei^1ZV)<4{z-`=NxlORVud}a>yy~ki$#mfB2uIyQh0F0}QZSNiP+Z%S#MqdS-fhdirY`4<0=Du#Q$MZ?m1QlG$(*oyA@>o5e-p&9e0-%9A2X$KEv0E{iz#mT@$X^TJC~?|uC9HZICy;ANXK$<k=$eTY|anrzpj2M-?f7J0S?80l=A=W$w&7Ta>0$Ix`W$@0=WUgiMnU6!q0Ud6L*ndJj7DnjgZ6_@c`w=d(Xa+OS_UAkSZxU|Pbv+{Zq7xHxWOR|yQKf&PT_c|(<@;fUO_zjd9u-_R~@>qTq%Wavg^wV~_$umTdF1gk}%Ve#al{CR5JyTt6^A+?o0*n^1^ao`>Wl1`~AM<#TM`tqY)heDLQZ>4cvMtORa0K<D?W&w7v$EIw{^k1*uin1#CZ4~^lPpil>miJ88V@fX`@L^pK7Rof;O$M%gMa>WSlCVJk3ApJx3?_Ut3kBcz`{q^+1?eFd~sDTU9X=0JlP+8HF&u9kbXV!2PVMb>nJ_j0_7m&UzWq8H-lf6+$2uz7T;~oav-7MtF(Ys&*Pzx5w<;mTDQGk59a8lae0~L=V6@ZSstDz={%U_(d7zp5MRyuV}du2%P3jF42v=kuz0k}F5^7t)7xa>B}I}JWt7fhd_C|O#wh1*M^G9Q{T8DSA6~!2-WWFi%VQwg)bopCrAFd^4B(?IO<_HcfjY8n=>R`d6~a>0_^3|a1GTmHfV9{EAEnm;?PW&eBFjB`4*WD)$FN(o)fS<vsSiB=3@Xm&d6F(NZvjo>IhLwlUYU8+<RbR_jv>s`qJ~2l)jET{o)nA#0A~ybM&r*`SplH=Jq*RWOv)up0=wG4yknz9D`KHts328+8z2ExT<!`-RsNV2YF`~|VH|xj+SJaNErDUj=~)~iw+@&k54>qMzh>CNreWuw%Z?a(7qr0V1NZ?`<vOFJjMqibzomv$7n&iglkvg+;g{Y+5C5E68*7$j=Sd8xSY^>1s(%x$idaw*z~BNJjQ7lYa`4q5(AnIZ`txkIUE@ihS`)qrs743=!@9xLUuBB>j8q?0Wuyx0^=1{#3}7bZ5i?x@{}L{W{7t_t^XswIb(zO8Oa?2Buqp5qVBcygR>ccg_2~-h`kp^R79L>R(G{L2fxowhP4@PX$p-zCG%EunkOI!Q@wk}f$)@bnJZo>t>s8!8g}pPvo;QKrX&vW?WgZA3`rvS1S64LW1>Z-jZA>g}pk@`*FyG-Bl-F6EtB>P+8=EvK#5Mvx5cspjaGjLNSyU=E;A`&dO#n5TPJOEnTEAyyQm*2!f7%wM7kTRp>7s=osQxC-fp*uh93Gt;aL{=dah@#rkm_p7eMMynov7{p^ES!j8c|tu@AF>E;Z%-`Wmv#@4^4wbT+Wtw1Z@F>A#A@?wlj!4FMy9lIq<<{%*@f`TN@B{yvrpNdmH2-`327A+vDSR9z%yi3{XkJ6GDVq7fH6oUCiB%WEkcygq>e)cWJ|W4F7JfgFc82ATI>^EjLhLfSr{v6vZvZ9<ulnPE)Bo@Xq^+aj27l(0+~UJ&W@a|3=$#iGRx+mNSgg3!k|Oh$2@sK#+Q(H4YR;h5gv0;cAUhs#eXxBaps@q7gB7M$0^2X@nu4SP9yAwhGs6NaZI%(P+Dw!=eN=j5+JDyfyG1KD_b2gB2b=M*;!KrlOWx8-gqoL*Pfd4x>lv2)67xgeqM{A<iBvVolGcLLseed<};JG7DT{p$9n`Di%>NX)w%z(NH$bWt5a5z=VAS{IwF<^u~!V01GI_M)OsY#vt^=zFco)8x_zgtc@I_K#bvHH4muOi4StJe>$Me6&A3Q(sFe@K_=Cw9n6iESy5Uo)J)Ut5+LJIJ6dPJdb2c{@uK6X;oUmA3JziQWK7-{)p-v5I$MvnX@Yzx*dKVK$9+AmQ3|X+j7s8*IE8_?-}i>trItjgk1xE@f%o-9PhZukFwCPS<b{AwIQW52o#IJz1pH&3U8Z}}Xm*}07R2S_c>z3X9VIC;6BatW>2`i5GK#^yt5e$QhL#MtYL?Cm>M93aV;*4fz<Z>|PgpBfalE1OzIiCnAtCIN1^|mc%ksHn^@|AAJ5U+7rO87F9!o$29y{!zfyK%7Oo|N(%An(9Wig={4LsFdI8XA4uYMPP#g;!TBS4}@sx1$Ms2<icLXj_t&QhS5WLD6jjNBVQY=H3sk;GHjDw-uv<Fh1fBO<+~6R!6KPll0&;}YH{cmR%{!|4g6GZ9MLmNR77B0x?~PVoWe2J@yT2VUTmg^qFZtW4?l;X0e+3&I!hW|;VlPQFDAmtG901R<UJ36(j31k8{WO5fEjufmOwouC4@iBr^g+rb0ILRfZCMY0g(hEUlWxDRHIY%xHsj?l!r_*2;TJy^3P79yR5$P{7!gRK3eCNI(4V4tp{YZ%cRP?W6K4J?csNNfn94$wm2C%lZbNoD$~-X??P0(Pq~*Ka^%iBM?l2>+4**wT5kKM@P8lNAnq)ju4=^B5#qAiSH~z8hwg6gGMgxx#5RiVrHL%I^x;NC*AqKmoh;eQ2`Qar-D7cnEyaCqn^%BF%e?qXHV25y&{eO{Ou7KK6)%`VM?wC<4AA(j5y)qp+kV1!eq;%CL&L|8IM(OCnqRbOc5k*l+5jYYu@~OB=Vc76Q&wu*JAYxRXXvUfovmr&2*p+GHM-p?a)QaInuhCI4|U`uqH&O)@bz%%?)+(`cekC83Fl{Ak@`LBj&TH1tvo0R!*t2c>+0!4sRgky|YaRB_gzg%=J-CW48W!A{1H_xDc+p0+-K`6DWd30g{(oh%H|#1n=);HomgP42SFIcO^Z%kZj&fLIQJHQV5!&=f)%8+<5qtwoa>OAxdrbbJ#*xA8cj>qx2bSktiNFpM3og<Uh#ov1m<ld}n(7BVK3M1(g!YjQ+<{i&k18!30|yRD6R&fHcR%Blep2DCxJ^$*SZN!n6D@jr{&6-f#W=@Op=CPw5$|LV6PU2P+>Xw>BHPDBYiZb<4aGW<P{BWtJm12S+kUGnbkEBh7~Ho#X#fd&-=tnG}vF{7iXh|w0Gg4U8;#4k`dwRGK9hO;coO{h+Qi5vv_3}p}#G$?eM9lN;?td6!{D(eHee$sM?j0_KGF$Cgow5h~orY77EqeZD6>xL?L&LcRs9g+&5Dnu(DpzR$Q^Ppf)pSgW+&s#jW5wSFZWxWk=1i$m-3~DKR;qB<`mj?~#QZp*EHHg>K>oP6^>Idz@09?8skQ-}+-Y~#vK`;m28rk7wzkkQd2#|^V7=dGLlNHGoDHc#zEThB6zn!=S-U7SM%G8R}ntZwcWZ->y_!a5kK$zY${rZ{r+x`7s+n%#wc5}RO+wT2KaKRlI(I@TvDR^Vfd+prKwY8Y=LnQo(WrjWCEpg0S!biIVp1L8(o)ZzfIyTxT(Mu=bIy=w^vdhInV=SvdUyt@dCwR1fi2r+p|N9&K-xFm*w=nTqn1N$$$wC7Nfn{R1YbB8~kdq3)X}?1NXI;FKz;Ary#Fm&}X7DI|!n3iY@Iw7?u%AlxmA1W|h*R6zk==Lshw6An$>*@PXCJ{cn$rYM6Pds9eh;P~GL?`4n+_=@{o|q}AfbqlgpBXX-l7aGNin*I7R*Q|D743f1_3>JG%J}iN<DLzpd26yv7!$Jg|<kW_^zdN_jooSMEANylM4OdOeCM?L>HNYNec_a1*z)yc9zT+io|Z1w8F~!g3Dad2iak2m3`Ma*d~}zOOZ+zKCQn71gd)AKdpQ;Rli7!uA#tM*u8_B@S~TxgQo&mwEhU(pSRHp28hM|0h9j3=iX5C>$%SD#VR@rg#iSr1G}cp+Hr<8i&2ODfo*TA1cVhHK&#!Rp*q^L<q1KJksKg(1@T}XWyqd+DC^qfziS#q#=V^+a<Nf_%4!m7tBG&YKbh@&uC2G*-25%D$PH~UO^#}g3;(96=iZLJ8z{K-t(n`9;nYlurPCZ+Q&A_a>M5CiQ8{H>CWE<z?GT3;k7Ubx+H&oXEJs-Vfl$3=b5})dbuVHZ*p@7*i|r~DakUbL$cCX{10@vM7L34XV!z9Bj;c2JXYmb4*4JLXRq+?>dj;##;nfvN2v4=ca~aRhH*mVb5daHy?XgK%o6yxzXapi-BQ8c9IfA&RLRG~m7e9)$oggSYr;uZEbwHxOj=)C)(<khKOFAjS1q{vfW2sG8dB+2=^YIKB&Ke!lQ&hu&5|aY`VQ_veK%&FbS);+~Ys7nY9p&c~M1kYm^L)WxXpc<IXO2GU@|sMAs_zh>lZcQD4p6mGYX{{gCxs#pz#<#QPpiNkuIF$P_|;Mal<MJD@T!JAqUqXRzvzO@h(gNfA6JyDnpix9LW-~fow(pR{;Nq%0hB3=2H1Bo$gZ`40|ZWw`BSt>0CA@AB`t{Jo>_i9GMG1S7aEaFbOjR(0<}059zaA`Wz)ca$aJ$Cxo{{!(KU`dMvn(9*LecsB4QF<Iwz;v{i+ttSiY)jRz)l7mQC8qE_yZLj^z2~qSC^u0f`iA0;`9f)I?70%xcmE_GeK9ACpP~1K54xI;sRIxKCta4PapVe(p^quU_nZ_x`oFP6~+ooGne7sE>)IavFmn0U&EJt2O01huFmIh$U*P8&d-yRe8U!WbMzlDyXRmXRSD*R$N%S12{FVYLH6>0~sd-!$fEn0(YG-qn#H5ASv<%ODt@*0`|Qu4r~*1L5P)-y(5Du2h|YJk~8(mOb+ZYj@EYP>}^o5Mi}83)O`!&Ix^>2#io7Ta736f1g46Mgkd64X~X{My-=}?isfFB{1Ug(WKC@|NvEMNlb6kk`1^F~`HlBZg(ZQk4*pRw#fWDQqC2yvUIr>(eu)#+GQlba=301jV9F~BLf~~b)I5q>gg_+iCJa=}$hfYJ8a8;UF(78`(B>DUpC+q0a!so0*4OM_sfCS(c;)OCo%C*(BURvKkqy;iN`iRoGql54yOG-8wHYnlILguW2r$%<b#DN?vt=|b`&w9+>aJeN4qJ6Ai%!zl<Wpw_H6CF8tMLOF_#NE9Hd%77_u&pToFS(WIBnb_s8IuR24$M6v62cFibT~S3OD&g{C4w~JB`q4+3RL!f!<|$Zf?a9{l6h@xc50V9e4rmINt7^mf4Q|Y`?5)=MGJ*E}-8jT-myq5k9A!6QX!AAhkb%Z29&~y%<7)V{p7IF@D3rM)(BSWSfBE?9i3j@BL*R6(3`KF;6OOcUB^fEyMLb!ZP#lp@d2)F*-y`>%>%!dTq96zNX`juIe&*_u5^42AQb#iwAY;*>e$2lMJoh$I8e(oj<J<eQ&EmrKYqC4x$E9{F*3ndRY>*iLj!h{2y4t(g^xo>=kG=jwqFKpn9`R9!iJn^(c_RBTLx3JUc@SX@kW(2-q*h6an%-UY}ZnT@A==rA*UP<cwZrEKy^?NM(Yg%@ioIgdy02$}og@*VSUcL3S`SuyM?{E1cFUI|GITi$AgDcnnQK3zUA}mv{smA8ZW$0weeDk|V3i+AQHAt>WQ~0OK>jp~38NxFbDUd@zPg&lSW;=d^5)`feT`t60E@<|kMeU}U%gRJLz7R+>+Y{%F3-7Su)zuH18sR+4*jBk;zYL#1jxitQTc<C@mPz&7<6kO^kKU2h8T$EZo68O6kfuV)}{bU~M;-Pfp*W(KbK@{_z#Pu<WM6VOD(f|}^*-y)$E;F}MlVwNN-#1%byt8K9~jHVUZY_!|cs*uMaxr&tGJTeVDZ1crpOC`2FC1l#GSO8RZ8qN*xow6dQpt7JoP3C@RWpJFRpis_0ynWJgL_t0U3W9;euJVae`33zX1wkX3({Tes-KIsf0JOpVG}y^b;fIDGr||;xEsUiO)i?s4U%#f9YTB$RTK10kV&WY%-^;3=s2``zY83Q?-8W?(bBi_F-H-0c*MtGOfGRPT?+_#Bs}O0l7Yb!?3ECu+x%q-DJG;PC0g<{L$qED671}qt6DTyPn6NAp%Ybem<%L-R3*d?|u%8_V%fd+Ixz(U*Z|(;U{YXw|>IGT{cLQidJsHr9(!MfdA%*P@umQ^ia~}pJCVP9Y{B`3Hg42RFckPs!)VOWS9?|E5aaJf-PW`0l+R)8$e%)lSJB&cx(L|}@PDczlU@*oDBO@LUIE?jRq$5;{$s_BMM6V-KH8|XdhF?A&_2ml1<lqSw!JU>6Ow$w89tQ6F6TP<Z^CDDN`wQK==`Dger7Ze#EkrFkF^Cr2G7^fZhV#d~k4$pYMO|LHKskw&E%G4*jJ66@-S+w)RCP1PsL;YxU6d+BmSQ6SqnPyeLmUha(=-{OAud~K3koJ00Q(72U`bO^<3}2=Ch)L!;iY^0d~;Y$fMd~R_^KazBQr535Y$3j%{w=sMd?9T<<85eJZli0VenO5R{qs-{3FHM{fNGk&dF5QFARB>PI4GYI>g1f=8zm?B2CQFexV7d3mDJ5{jP3bP$dV}u+ua?i)xtTjbjE9tINqmq5QT^r_$(|bz`A97^&AT&?7YLQ${2=mn!Z28&wi;fI2b{zAmEqhx|Z2I<U7i!UrRAzv+gAy^8U>a0qaE31zB)TgzgrsxB*_XEj9AQxKAuH4Gi+md~}jnoCIq1|$VdXVyI(M+agTAA9<uk8fUpLeb|haRth^aSGj7*K#nONxDeR+PYHwvduH==uYwkjFl5pI1`$f)Rn)nzCR}yO^WKK36OzL6oQ(E`<{fqxrXct<bS>{MobCN3@+K1c=3^6$fI^1iz!XAkSL7t^u-e|UP#M2T6qq40D&x6Xtis5y<-?2!SHw$c6A)LTM9eIHqZm(tH~<vfxNx<@bvC%z%gwN9X{TjJ+-!HRGhH#qzriX^wPT5=150~bUA68ky8>#Vi~BqxV3>>y%zlIWX$zX<(@iOf6z8z+LZ$HLM4g<+`Ml{HEINB5D&DWo9$J4o`8(mj<shfh63)s^lag__13f0V$H0IHdbRgLExt@tf8E$8ihYW<O5YVR`568TRib*bSUxQ!7&IVaR&23m7ZIVDc+N%62?F9I#^%8-IngkEfbCk#h6gKTW%T9E7Z^+>US(J>FUhqx><a;7E}|G?MF)rHo`~h8%ayW%{`%Cp$)}usO^xh3Ty~PLR5>@(>hIc>uj5bD1fT~Gn8Oi8doQdkv$8HumeXHE|P3SE8)N7bHn@<#>H69<^}8xUyL9v7DO!u6+I2qTB_6$^i!{Ip&dl0?a5fMnd$`FnVG=??%U-SrZ?yhg*|A9u+B?p8rQ5VTGr;1w)3gMnr%COBNVH_V0Qwex%nw8lx;>Uw&H6G_<H~!drGZl+mU1iAW1(RMwGJ)%GqXz0hf@u$;#bkTLqi3GL>QQ^P21$nUVB;)MRmIFixYf*`RI`@g+n#tfi@`te18;(7m8D1ndV6H_*P$#7?BrGgWq~8h$McmW;=&4*Zog5j)VxtU)nFjnPWPw^8%fcL3L_j*^9C^yausjm!#DL9W8s^>q%TX0j_%@ItCg@7VTL!$jS_CcR2+Tpc0h=Hlq)6p=1%aAe!`95QaC6+l;~$_&?U2dgUd>Xa&}>mV&h$UD~Fp<MZn-E1y|j`78efC4mR<`QUA5}RYd-UI3mcZOmSJh6TXyAQI4RO#B_R`!b0_$-nf)AmHOWytIs<nxfP71CaZQTp8~y+W$@yZ3K@`0(<5c>MCiarlR~FJ7AZG!6wl0yv5WW?4v{qTdj;&|L9D(1(bl=9C~n&GD8oi9CL1@mkBl)Pu^?;*5=wI5@}I-3XbgQIE+S+u5Nr^zeSn(K#4eMx8T*;nQz$W6t9%pR#i8n}gR&f(uM6#YMRk=%TCMK`o@)QJsb&g6rjyLp49e3QX<<SQ5rD4+z$P1`z5B<9ZwX%rU$_F^<3&nYu&+%Gs83Z9<#9gzD)VX@wG-@B))c0>fST1!v?^XPTUaRWpm;FoGGEP6jq^7iYK%v?qTJN)<(b?N5Au1w?54?#$)(`}mrF;hjHsBDWfKsFYBc<4E<O=tGs(l6D|rn3dKsm;~k=ANxjKOCAYEAP^}A1wn<&auzCe7+`dWYV$&nJH9KbSiwX+&48p6JvN4E)}r-&HCI$FL_yT3ldA>-sq#G0UhSXkpJJ=3qNXo=)u3ZV3PQ?KU=cGq=5)=I;lb&N?Df;WSc4QnX+uy=&;vORB;MFQK6aNG94ETmE^}P>YaVp9TVB88{J-{XFV_)TMF2JUAc}1ey%P@!?iedT08lh`W(yTO)bGKYeMz1Ay}s5Vsl<fQ6p%ZfuA=nZ7eT5fTxu`jxQ*GT(FI;8$3zF0QI6@in(<q;nN%&rj#n929N<n{aiSzB&9h+Q$N8WCEnRx^?e(vJ`iIgxU;g<&z}~U?^*{gh+WYT+{GWgS^-upY^U`1c^iK)Yfj84X|8oLYG=Kfi|0+jz)=}-RM|9R`l5C-<FhX=z4!d5sj!GQ|#t$k4-33+2I8;Vs%Bp~%Z)lmun%9o#85DGo1dURP>i%2%d6YmJ#rzm5dShVma$(T`;OUr3zA7qd0|vSr1<qO;Q{&cbK<#XDwbN6wO|_Oo(h(6!a^_AMD{16dTP0QWuCuKN$;!Ot7VZk^8Wv|Ej}#VSuA_s;`^`(EwMAp&Z)Gx3t5ma?JVcXLl_GfHnOWiLcjhVYL?H?zuqn6#XQW2aDZUjfeXq~a_A;I_(G=SaHX3acY5qezeL?SjfT8quNNMCwu_n3Eq))#q>l)8f94C&9U^VZ{h(tEJmgtUkl<CE#+CC<h1M{KLgxHM9t6Vhz-dP~Th8{knRznCe0Jn^mrH%T^ITICiu`VMcw@Oyx8b_T^T!NemVF!>67xc!id03PY2tAt=+lNezkhDE$a;`h)>Yk)xiz-(lTsuu9<%5;brg58;b1?9H`IfShTdlK8YS4pWwh}R>!CGhYO*mW6!PCKF+BaDB7~+YQlysuxAu2fK2mF!Dtz$*a%@iO`DGjDW`0gaR>Kv=i+EYU|@71pCFfdvbzHJ0C+n{2AR8}sAPy9RLob_J**f{6U7=>#u`e(|$f<fVW##-su@!~B3-q8*7t=rKiJ*#HRGsBjBOcjgA>iQgPkAZq9I8=j-j$w9Qi{b&vN4OZo;2OgSaE}|X`|iC{k8>yP08a*Lb_zhRzFP3qqtGr0a#7^sq`!}dH*dyPci;HeGvv*~H_*|{;1=Vp>-Nq3ivmvxyOdG}e|20FySa5<1G_UNg<5di$c8~ZKPh8smzTs$I`KeR6<gI2BM-%chKq#FrtqJUtb&bIf(hH;b~<iN%TlW-+L^15Ps%!=W$^z$DF=*<c<o67R8YRtDETqA^(vRJq=s*|Yjr9>t%_cow$lhqy7QK;m87cP`ti<Jbs?3Ng!ggQT6=TrGo9k#`o^UM%mf8KNlNS|U((@QG#=g^0UblxWcR?Amcz4r6vgn-ST_??|CnEgY)BG^ZTZAxU9duPRdhJMs1H@Ooh~|KGr8qlHEqA+%%sBkKrznWn1kRiF#Ih8re)1Y&*`%inU?8PFkc0PiwATMOLSNWMXO>$<>C^2Z6sHSbD-<S@G5wZC_3jNi7(;dkRF&pj=h6hnsYVzMno{ozR5>v3PvlZ(*)WMQPFDxXU_!Dy`EgMFwsOoYRaae(VgYB*>ioo*l53C&fu`feE<Ay_~N^_M=I5xR;CzI#Yv>oZU|PYYT{GoD)>j=e)l{)`u6$X`?f{C)q;+O9i4t!Ij?^I?b|nh7wS&?&R*4q1`!WMp->s9&|i)m%8J{qHuNhDri5>1v2)BjqDCOnP@Zx4%2N)X&4R6kX9|2BX&T`dY+i$@^GatF_b^9&VJ1as#KXdfPz%P0yq4%Iig2ep3t=;ZFb3$WamIL6ZFY_ZYRn&UdTToHOn8>hVuj<i89$Dvl65dAeX1@9>+$Vy*;3zyR4A6<DFwaXbh2Wx`ZSHkt9;Lj0@k27D$j=pqy15ljrL)N0hdA1aKHTRmy{;5DltR#rS&vVAU0%p*aX6_4jZt{7j=+nQ9J@b0tO#}6%F{^CjZiU3D5Y}f^arn<46vq7Cawl5Lf3@wz-&t)usClXn#Z{2IEU(3+R<Du3-Oh9-+U)uBXZlns67%6&S*y?jv27imK`U5kH)eW0gZ)g^CUv(7*!<0*+Nx6&)IgR|R#1&oHROY={L$GkUBy+CVfQ<oiqVtU*2Yt6_!F+v+qC*SMxj)7LGmLA#5@Wu$7;ePP|O{uLreE<E*#yV22$*RKNxQOjeXq*=}^-2}h`60x`egZCR(Q?sll;h1aB<O%X#Wkpv;1cua@Qle33k21nk=riyu(ye{^=H3WCmoo!igj^J(TQO5x;{`SGorw+Mwb=KmR$F-Dd&P^`nf(Wwt?M~I^>`e6x?67xGA&4MDA_@XF~k}m!Kp7o@g)280IwK!h)ov~F-D%Oc$>@v@|*2DE}LxUSpz#*=sR;^6N87hU8(Txx%1KHntqk*D9<RJ1SLZRk$t|x&`Ipam2se&FxCL<4AWXhRQO;JY_xQ(@?Mhd7Er;Z1MkJl-@SVNCj938+c(EA-@KT>HzZPwk<V3VA|U+rQTY7z>xmCXe9yKsqTT}maeNR4kmG=Kh(bpNx*}F3u%P<VBSzH)JdrOv14>cO@$!vP^WCd=FR2{V7_<Dt@r$?L9q*o)9>qej%AjQoR%GIfv4d13QrFeT&vpk3Se+R79@I14H`h;v7xLlF;?X=tCIaKa&k}((5Cm7s)Zs^dbGQ^d2$kGK2rrb75a>^-H4)v%jRhA=FqVho2o&a7Z06U_J;3!9e?t@F?wi5d0m!n6ib#QFeNF9o=a|6h3;NOcv__I9Ii3*9^OYq48OQMFpV5W{7q@CmgT=Qd+o&BWPR1(r7>afDoN(h)H&!%rYi@@VVt{xxfIdCq4+Q>c>$bS>R~cG&QRnWsZ!e9_)V@g`?YCr90fB~`K;)SRUr&?+W6u2}DIfn>JPdyP@I@b>Yn<Bj(v~Wvs$P+>x-dz%D0I)((}?11g98<R0m>lQBmDNPDnef(9q@B|a`t8dYj@0k*Y0|g(fKS?S&<sjDsC&MoMUW-B15YeT=6Vrz(wt_b!;|%q}@i*Z^L1l4Y_oPIw2&CNa<eVLxFlas7x4jZOEZxol^XYdT3}*A}Oid^rs!NmF8d0hOV3u9o!4gZ^&W^N^6J2($baX>uBnL<W=)`HK{5euWeG*ACle0mfHV}I;Q?2(I<hjRaLqF8{^ll#N2A%|9xm)P~c0me6mi#eE~OR<sxn69iHIT-uBD0yVx(9?sxIDYcG7KhhD)R(U&V{`Wx0C)T0_1`~N2u#E~Myx9Sa(HrE?LXk#dcd;mKUukoQU37?s{^t-FVH-nLchx7PiSS;7WErrc>w^3gkh9=YP8N>vf4L8Z=t~SaiJ;cOXlY_%2`-89c_p2zQY8fjUZ-az&hib`j+toj)f-zf$Eru93bpGhjzZ*1gumV{BG`p&XW)V~qgeni^^BL@~w;I`qse%`vBGC33`f>djO8I-vQV}B^X3-{^CgfK0&oYqxK+HpoqbZ!i4K4RE@F9@j<?C;i?0@2aco`l&Kl=7%`0n}fw>qi-$)4SqxJ5@ot0Aj`RX#i?#N$_gcnMO3;$9%6`TXHHMb;CcXiDA#J#A2u(SA-Aftk2T_HC0mHYgK^=a=w(JYomv_mIR7hD>wx{^j%Im*JbY;SaCgym<S=ZG#jhniP~_Q=%Z%ARN0b;K@w*??4u3!haocEgv7J(uhfr7|}STy!Lk3=H0B;5WR%m-dHHsHLIJ-!lH!kP%$w>1i$B+wZ*cp7<Y%SvG*zi`zkb2pAW2qQ^e6V9<IR&Jbe?fc&pcVG;mhO5G>M`qci1h?KTCa{alO=O&>l_!eI05A3Wf}gSKzpp;D)1Eg%r~^C~%8mh}_vp%9ujnt!h%+{9K+mS*{iULhAWaq4I`$GB|m14A=I<^2+x*EfS4R7?>B?gLmKo8};$kyjL&sv`~t{1s^eG}QBcwx^1pVa&|eJUNnxp6yYQ?+T+(-^4%;UjygwEY|5J>bs`%M0xR$fJWUil9_(qZ!DzVaaBSdmhjx)DcsSQ{_*4JIdW&=ptR7aQ`X$+gIAx#YoEn^2HMzy9Yb}r`RwD~iBuHMo}wiKZ?t+Y;Z`Y<wL=a_N{NN?DZ<8uN=@9(h(l&2W^Np}Xl$m&`6wCL4g;JVV2LBK&Y>%9O*2moaGmF{uRu;&K`4E;!VidfBBg3xroMOg)KkJjK@q$K%Hyb)6WTN{<10*^jz38rs8ie2a*XH%HdgAu=A|r&*>5DgY0n;LBrPZ!H(B+azq)E<XH9Lr$l86=-tZ`Q-n5sr_VqjT>VW$!XGHA^`Q(liOtvY=qEYLqMyySl>P{{y!DTo9C(Pv@BAbI7#`d={cj6DNiS1%_+iB;#V+JuaKIhz#hv6Qd&0PXjXOutFVUcxx)cB5%8CUlc?-jP+N>TSPgIWyJ8TN07!#mKRKZMr!+#|<o@V7qbgO1gGREmAUXcG0|6S}EaI6up%V?Z21Yz9w6RjK=nfo{vC@p*ele-LmQr9%)A_05N_Z%ftIte7xbGy>Iaq6soKe$@%@FA@}Z53||*(_r5#>4l?hBkzTyj<HB8^_N=`ai?`?GMoPez6po-'


import base64
import hashlib
import importlib.util
import zlib

def sha256(body):
    return hashlib.sha256(body).hexdigest()

sample_bytes = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in SAMPLES).encode('utf-8')
crawler_bytes = zlib.decompress(base64.b85decode(CRAWLER_PAYLOAD))
if sha256(crawler_bytes) != CRAWLER_SHA256:
    raise ValueError('Snapshot crawler bị hỏng.')
if (len(SAMPLES) != EXPECTED_URL_COUNT or len({r['domain'] for r in SAMPLES}) != EXPECTED_DOMAIN_COUNT
        or len({r['url'] for r in SAMPLES}) != len(SAMPLES) or len({r['id'] for r in SAMPLES}) != len(SAMPLES)):
    raise ValueError('Danh sách mẫu thiếu hoặc trùng ID/URL.')
manifest_path = RUN_DIR / 'manifest.json'
sample_path = RUN_DIR / 'sample.jsonl'
crawler_path = RUN_DIR / 'crawler_snapshot.py'
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (saved.get('probe_kind') != PROBE_KIND
            or saved.get('sample_sha256') != sha256(sample_bytes)
            or saved.get('source_sha256') != SOURCE_SHA256):
        raise ValueError('Run cũ không khớp mẫu của notebook này.')
    if not sample_path.exists() or sample_path.read_bytes() != sample_bytes:
        raise ValueError('Danh sách mẫu của run cũ đã thay đổi.')
    if not crawler_path.exists() or sha256(crawler_path.read_bytes()) != CRAWLER_SHA256:
        raise ValueError('Snapshot khác run cũ; hãy tạo run mới.')
    if json.loads((RUN_DIR / 'source_profile.json').read_text(encoding='utf-8')) != SOURCE_PROFILE:
        raise ValueError('Source profile của run cũ đã thay đổi.')
else:
    sample_path.write_bytes(sample_bytes)
    crawler_path.write_bytes(crawler_bytes)
    (RUN_DIR / 'source_profile.json').write_text(
        json.dumps(SOURCE_PROFILE, ensure_ascii=False, indent=2), encoding='utf-8')
    manifest_path.write_text(json.dumps({
        'created_at': datetime.now(timezone.utc).isoformat(),
        'probe_kind': PROBE_KIND, 'dataset': 'AIGuruTinix/ViBioMIR',
        'source': 'embedded samples from data/links_corpus.parquet',
        'source_sha256': SOURCE_SHA256, 'source_rows': SOURCE_PROFILE['rows'],
        'sampling': SAMPLING_METHOD, 'seed': 42, 'selected': len(SAMPLES),
        'selected_domains': EXPECTED_DOMAIN_COUNT,
        'baseline_archive': BASELINE_ARCHIVE, 'baseline_archive_sha256': BASELINE_ARCHIVE_SHA256,
        'sample_sha256': sha256(sample_bytes),
    }, ensure_ascii=False, indent=2), encoding='utf-8')

spec = importlib.util.spec_from_file_location('domain_probe_crawler', crawler_path)
crawler = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = crawler
spec.loader.exec_module(crawler)
priority_path = RUN_DIR / 'priority_probe_snapshot.py'
priority_bytes = zlib.decompress(base64.b85decode(PRIORITY_PAYLOAD))
if sha256(priority_bytes) != PRIORITY_SHA256:
    raise ValueError('Priority probe snapshot bị hỏng.')
if priority_path.exists() and sha256(priority_path.read_bytes()) != PRIORITY_SHA256:
    raise ValueError('Priority probe khác run cũ; tạo run mới.')
priority_path.write_bytes(priority_bytes)
priority_spec = importlib.util.spec_from_file_location('domain_priority_probe', priority_path)
priority_probe = importlib.util.module_from_spec(priority_spec)
sys.modules[priority_spec.name] = priority_probe
priority_spec.loader.exec_module(priority_probe)
if not isinstance(USER_AGENT, str) or not USER_AGENT.strip() or any(c in USER_AGENT for c in ('\r', '\n')):
    raise ValueError('USER_AGENT phải là chuỗi header hợp lệ.')
crawler.UA = USER_AGENT
crawler.ROBOT_AGENT = USER_AGENT.split('/', 1)[0]
(RUN_DIR / 'baseline_domains.json').write_text(json.dumps(BASELINE_DOMAINS, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã nạp {len(SAMPLES)} URL mẫu và crawler {crawler.VERSION}.')


## 4. Chuẩn bị bảng kết quả và phân loại lỗi

Output riêng: `vao_duoc.csv`, `khong_vao_duoc.csv`, `chua_chay.csv`,
`loi_theo_loai.csv`, `can_xem_noi_dung.csv`; bảng tổng là `ket_qua.csv`.
Các CSV dùng UTF-8 BOM để mở được tiếng Việt trong Excel.
`error_detail` giữ nguyên lỗi kỹ thuật; `reason` diễn giải tiếng Việt.
Mở `ket_qua.html` để lọc theo domain hoặc trạng thái.
`uu_tien_fix_domain.csv/html` xếp hostname còn lỗi theo số URL dataset giảm dần.
`tong_hop_domain.csv` gồm cả hostname đã phục hồi; `WEB_ERRORS.md` là báo cáo lỗi chi tiết.


In [ ]:
import csv
import html
import shutil
from collections import Counter
from IPython.display import HTML, FileLink, display

STATUS_REASONS = {
    'robots_unknown': 'Không xác định được chính sách robots.txt; crawler dừng trước khi tải trang.',
    'robots_denied': 'robots.txt trả HTTP 401/403; crawler dừng theo chính sách robots.',
    'robots_disallowed': 'robots.txt không cho phép crawler tải URL này.',
    'http_error': 'Trang đích trả HTTP ngoài khoảng 200–299.',
    'network_error': 'Không tải được trang do lỗi kết nối sau retry.',
    'unsafe_address': 'Địa chỉ đích không phải IP public được crawler cho phép.',
    'invalid_url': 'URL không hợp lệ hoặc dùng cổng ngoài 80/443.',
    'oversized': 'Phản hồi vượt giới hạn kích thước của crawler.',
    'response_deadline': 'Tải phản hồi vượt giới hạn thời gian tổng.',
    'rate_limited_deferred': 'Server yêu cầu chờ Retry-After quá lâu; crawler hoãn URL.',
    'redirect_limit': 'Vượt số lần redirect cho phép.',
    'unsupported_content_type': 'Đã tải phản hồi nhưng định dạng không được hỗ trợ.',
    'unexpected_error': 'Lỗi ngoài dự kiến; xem error_detail.',
}

def classify_error(status, detail, http_status):
    lower = detail.lower()
    if status == 'http_error':
        return f'page_http_{http_status}'
    if status == 'robots_disallowed':
        return 'robots_disallowed'
    if status.startswith('robots_') and 'robots http ' in lower:
        return 'robots_http_' + lower.split('robots http ', 1)[1].split()[0]
    if 'html instead of robots.txt' in lower:
        return 'robots_html_instead_of_txt'
    prefix = 'robots_' if status.startswith('robots_') else ''
    if 'waiting for domain lock' in lower:
        return prefix + 'domain_lock_budget'
    if 'request budget exhausted' in lower or 'retry budget exhausted' in lower:
        return prefix + 'budget_exhausted'
    if 'winerror 10013' in lower or 'network is unreachable' in lower:
        return prefix + 'environment_network_blocked'
    if any(s in lower for s in ('timed out', 'timeout')):
        return prefix + 'timeout'
    if any(s in lower for s in ('sslerror', 'certificate verify failed', 'ssl:')):
        return prefix + 'ssl_error'
    if any(s in lower for s in ('nameresolutionerror', 'getaddrinfo failed', 'name or service not known', 'failed to resolve')):
        return prefix + 'dns_error'
    if any(s in lower for s in ('connection refused', 'failed to establish a new connection', 'connection reset')):
        return prefix + 'connection_error'
    return status

def result_rows():
    rows = []
    for sample in sorted(SAMPLES, key=lambda s: (-SOURCE_PROFILE['host_counts'][s['domain']], s['domain'], s['sample_index'])):
        record_path = RUN_DIR / 'records' / f"{sample['id']}.json"
        record = json.loads(record_path.read_text(encoding='utf-8')) if record_path.exists() else {}
        if record and (record.get('id'), record.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Checkpoint không khớp ID/URL nguồn.')
        status = record.get('status', 'pending')
        page_http = record.get('http_status')
        robots = record.get('robots', {})
        # Robots failure may happen before fetch() attaches its evidence to the record.
        if status.startswith('robots_') and not robots:
            from urllib.parse import urlsplit
            target = record.get('requests', [{}])[-1].get('url', sample['url']) if record.get('requests') else sample['url']
            if record.get('redirects'):
                target = record['redirects'][-1]['to']
            p = urlsplit(target)
            robot_file = RUN_DIR / 'robots' / (sha256(f'{p.scheme}://{p.netloc}'.encode()) + '.json')
            if robot_file.exists():
                robots = json.loads(robot_file.read_text(encoding='utf-8'))
        detail = record.get('error', '')
        browser_file = RUN_DIR / 'browser_records' / f"{sample['id']}.json"
        browser = json.loads(browser_file.read_text(encoding='utf-8')) if browser_file.exists() else {}
        if browser and (browser.get('id'), browser.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Browser checkpoint không khớp ID/URL.')
        requests_access = ('CHUA_CHAY' if status == 'pending' else
                  'VAO_DUOC' if page_http is not None and 200 <= page_http < 300 else 'KHONG_VAO_DUOC')
        if DIRECT_PUBLIC_PROBE:
            access, effective_engine, effective = priority_probe.access_outcome(record, browser)
            if record.get('challenge_detected'):
                requests_access = 'KHONG_VAO_DUOC'
        else:
            access, effective_engine, effective = requests_access, 'requests', record
        flags = record.get('quality_flags', [])
        if status == 'pending':
            reason, error_type, stage = 'Chưa chạy URL này.', '', 'pending'
        elif status == 'extracted':
            stage = 'done'
            error_type = ('qc_' + record['quality_status'] if record.get('quality_status') in ('review', 'rejected') else '')
            reason = ('Tải được và nội dung đạt QC tự động.' if record.get('quality_status') == 'accepted'
                      else 'Tải được, nhưng nội dung cần xem lại hoặc bị QC loại: ' + ', '.join(flags))
        else:
            stage = ('robots' if status.startswith('robots_') else 'extraction' if access == 'VAO_DUOC' else 'fetch')
            error_type = record.get('error_kind') or classify_error(status, detail, page_http)
            reason = ('Trang trả HTML xác minh/JavaScript thay vì nội dung bài viết.'
                      if record.get('challenge_detected') else STATUS_REASONS.get(status, status))
        requests_reason = reason
        if effective_engine == 'selenium':
            stage = 'done_browser'
            reason = 'Selenium cải thiện tải/trích xuất so với Requests. QC: ' + effective.get('quality_status', 'unknown')
        rows.append({
            'domain': sample['domain'], 'dataset_url_count': SOURCE_PROFILE['host_counts'][sample['domain']],
            'sample_id': sample['id'], 'sample_url': sample['url'], 'access_result': access,
            'requests_access_result': requests_access, 'effective_engine': effective_engine,
            'effective_http_status': effective.get('http_status'),
            'effective_quality_status': effective.get('quality_status', ''),
            'effective_char_count': effective.get('char_count'),
            'effective_quality_flags': '; '.join(effective.get('quality_flags', [])),
            'http_status': page_http, 'robots_http_status': robots.get('status'),
            'robots_attempt_count': robots.get('attempt_count'),
            'robots_cache_hit': robots.get('cache_hit'),
            'robots_fetch_id': robots.get('fetch_id', ''),
            'robots_error_kind': robots.get('error_kind', ''),
            'robots_elapsed_seconds': robots.get('elapsed_seconds'),
            'robots_lock_wait_seconds': robots.get('lock_wait_seconds'),
            'crawl_status': status, 'failure_stage': stage, 'error_type': error_type,
            'reason': reason, 'requests_reason': requests_reason,
            'error_detail': detail, 'quality_status': record.get('quality_status', ''),
            'quality_flags': '; '.join(flags), 'char_count': record.get('char_count'),
            'title': record.get('title', ''), 'final_url': record.get('final_url', ''),
            'elapsed_seconds': record.get('elapsed_seconds'),
            'reused_from': record.get('reused_from', ''),
            'request_attempt_count': len(record.get('requests', [])),
            'retryable': record.get('retryable'), 'retry_deferred': record.get('retry_deferred', ''),
            'page_attempted': bool(record.get('requests')),
            'browser_status': browser.get('status', ''),
            'browser_http_status': browser.get('http_status'),
            'browser_quality_status': browser.get('quality_status', ''),
            'browser_char_count': browser.get('char_count'),
            'browser_final_url': browser.get('final_url', ''),
            'browser_error': browser.get('error', ''),
            'browser_attempt_count': browser.get('attempt_count'),
            'sample_role': sample['sample_role'], 'sample_index': sample['sample_index'],
        })
    return rows

def domain_rows(rows):
    result = []
    for domain in sorted(BASELINE_DOMAINS, key=lambda h: (-SOURCE_PROFILE['host_counts'][h], h)):
        selected = [r for r in rows if r['domain'] == domain]
        failed = [r for r in selected if r['access_result'] == 'KHONG_VAO_DUOC']
        content = [r for r in selected if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted']
        pending = sum(r['access_result'] == 'CHUA_CHAY' for r in selected)
        errors = Counter(r['error_type'] for r in failed)
        errors.update('qc_' + (r['effective_quality_status'] or 'unknown') for r in content)
        result.append({
            'domain': domain, 'dataset_url_count': SOURCE_PROFILE['host_counts'][domain],
            'baseline_error_type': BASELINE_DOMAINS[domain]['error_type'],
            'selected_urls': len(selected), 'additional_urls': sum(r['sample_role'] == 'additional' for r in selected),
            'missing_additional_urls': BASELINE_DOMAINS[domain]['shortfall'],
            'completed_urls': len(selected) - pending, 'pending_urls': pending,
            'page_not_attempted': sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in selected),
            'accessible_urls': sum(r['access_result'] == 'VAO_DUOC' for r in selected),
            'access_failures': len(failed), 'content_issues': len(content),
            'total_problem_urls': len(failed) + len(content),
            'additional_problem_urls': sum(r['sample_role'] == 'additional' for r in failed + content),
            'error_types': '; '.join(f'{kind}: {n}' for kind, n in sorted(errors.items())),
            'test_status': 'CHUA_CHAY' if pending == len(selected) else 'CHUA_HOAN_TAT' if pending else 'CON_LOI' if failed or content else 'DA_PHUC_HOI',
        })
    return result

def write_csv(name, rows, fields):
    temporary = RUN_DIR / (name + '.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(RUN_DIR / name)

def save_outputs(show=True):
    rows = result_rows()
    fields = list(rows[0])
    counts = Counter(r['access_result'] for r in rows)
    summary = {'selected': len(rows), 'vao_duoc': counts['VAO_DUOC'],
               'khong_vao_duoc': counts['KHONG_VAO_DUOC'], 'chua_chay': counts['CHUA_CHAY'],
               'crawl_status_counts': dict(Counter(r['crawl_status'] for r in rows)),
               'quality_status_counts': dict(Counter(r['quality_status'] for r in rows if r['quality_status'])),
               'effective_quality_status_counts': dict(Counter(r['effective_quality_status'] for r in rows if r['effective_quality_status'])),
               'created_at': datetime.now(timezone.utc).isoformat()}
    domains = domain_rows(rows)
    priority = [dict(priority_rank=i, **r) for i, r in enumerate((r for r in domains if r['total_problem_urls']), 1)]
    summary.update(hostnames=len(domains), domains_with_errors=len(priority),
                   priority_order='dataset_url_count descending', user_agent=USER_AGENT,
                   page_not_attempted=sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in rows),
                   additional_urls=sum(r['sample_role'] == 'additional' for r in rows))
    summary.update(request_mode='direct_public_page_probe' if DIRECT_PUBLIC_PROBE else 'robots_enforced',
                   requests_http_2xx=sum(r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' for r in rows),
                   browser_access_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] != 'VAO_DUOC' for r in rows),
                   browser_content_improved=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_tested=sum(bool(r['browser_status']) for r in rows),
                   browser_http_2xx=sum(r['browser_http_status'] is not None and 200 <= r['browser_http_status'] < 300 for r in rows))
    write_csv('tong_hop_domain.csv', domains, list(domains[0]))
    write_csv('uu_tien_fix_domain.csv', priority, ['priority_rank', *domains[0]])
    (RUN_DIR / 'uu_tien_fix_domain.json').write_text(json.dumps(priority, ensure_ascii=False, indent=2), encoding='utf-8')
    write_csv('ket_qua.csv', rows, fields)
    write_csv('requests_vs_selenium.csv', rows, fields)
    for value, filename in [('VAO_DUOC', 'vao_duoc.csv'), ('KHONG_VAO_DUOC', 'khong_vao_duoc.csv'), ('CHUA_CHAY', 'chua_chay.csv')]:
        write_csv(filename, [r for r in rows if r['access_result'] == value], fields)
    write_csv('can_xem_noi_dung.csv', [r for r in rows if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted'], fields)
    failures = Counter(r['error_type'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC')
    errors = [{'error_type': kind, 'domain_count': len({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}),
               'url_count': n,
               'domains': '; '.join(sorted({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}))}
              for kind, n in failures.most_common()]
    write_csv('loi_theo_loai.csv', errors, ['error_type', 'domain_count', 'url_count', 'domains'])
    (RUN_DIR / 'ket_qua.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    (RUN_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    md = ['# Các website còn lỗi sau lượt test bổ sung', '',
          f"Thời điểm UTC: {summary['created_at']}. Run: `{RUN_DIR.name}`.", '',
          f"Đã chạy {len(rows)-counts['CHUA_CHAY']}/{len(rows)} URL thuộc {len(domains)} hostname. "
          f"Tải được sau Requests/Selenium: {counts['VAO_DUOC']}; chưa tải được: {counts['KHONG_VAO_DUOC']}; chưa chạy: {counts['CHUA_CHAY']}.", '',
          f"Requests tải được: {summary['requests_http_2xx']}; chọn kết quả Selenium: {summary['browser_recovered']} "
          f"(phục hồi truy cập: {summary['browser_access_recovered']}; cải thiện nội dung: {summary['browser_content_improved']}). "
          'Các cột lỗi Requests vẫn giữ nguyên dù Selenium phục hồi thành công.', '',
          f"{summary['page_not_attempted']} URL đã xử lý nhưng chưa request bài viết vì dừng trước bước tải bài. "
          'Các trường hợp này được phân biệt với bài viết thực sự trả HTTP lỗi.', '',
          f'HTTP User-Agent: `{USER_AGENT}`.', '',
          f"Bộ mẫu có {len(rows)} URL thuộc {len(domains)} hostname, gồm {summary['additional_urls']} URL bổ sung. "
          'Gồm lỗi truy cập và lỗi nội dung/QC. Số URL dataset là số dòng của hostname, chưa loại URL trùng.', '',
          'Ưu tiên theo số URL dataset giảm dần; không suy ra toàn bộ URL của domain đều bị lỗi. '
          'Dừng ở robots là chưa request bài viết, không đồng nghĩa website không mở được trong trình duyệt.', '',
          '| Ưu tiên | Hostname | Số URL dataset | Mẫu đã chạy | Lỗi truy cập | Lỗi nội dung | Loại lỗi |',
          '|---:|---|---:|---:|---:|---:|---|']
    for item in priority:
        md.append(f"| {item['priority_rank']} | {item['domain']} | {item['dataset_url_count']:,} | {item['completed_urls']}/{item['selected_urls']} | {item['access_failures']} | {item['content_issues']} | {item['error_types']} |")
    md += ['', '## Domain đã phục hồi trong các mẫu', '']
    md += [f"- {r['domain']}: {r['accessible_urls']}/{r['selected_urls']} URL tải được và đạt QC." for r in domains if r['test_status'] == 'DA_PHUC_HOI'] or ['Chưa xác định domain phục hồi hoàn toàn.']
    md += ['', '## Domain thiếu URL bổ sung', '']
    md += [f"- {r['domain']}: lấy được {r['additional_urls']}/4 URL mới; dataset có {r['dataset_url_count']} dòng." for r in domains if r['missing_additional_urls']]
    md += ['', '## Chi tiết lỗi theo domain', '']
    for item in priority:
        md += [f"### {item['priority_rank']}. {item['domain']}", '',
               f"{item['dataset_url_count']:,} URL nguồn; {item['page_not_attempted']} mẫu đã xử lý nhưng chưa request bài viết. "
               f"Lỗi baseline: `{item['baseline_error_type']}`.", '']
        for row in rows:
            if row['domain'] != item['domain'] or row['access_result'] == 'CHUA_CHAY' or (row['access_result'] == 'VAO_DUOC' and row['effective_quality_status'] == 'accepted'):
                continue
            detail = (row['error_detail'] or row['quality_flags']).replace('\n', ' ').replace('`', "'")
            md += [f"- ID {row['sample_id']} ({row['sample_role']}): [URL](<{row['sample_url']}>) — `{row['error_type']}`. "
                   f"HTTP trang: {row['http_status']}; HTTP robots: {row['robots_http_status']}. {row['reason']} Lỗi gốc: `{detail}`."]
        md.append('')
    (RUN_DIR / 'WEB_ERRORS.md').write_text('\n'.join(md), encoding='utf-8')

    visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'sample_role', 'access_result',
                      'http_status', 'robots_http_status', 'robots_attempt_count', 'robots_cache_hit', 'error_type', 'reason', 'error_detail',
                      'quality_status', 'quality_flags', 'char_count']
    labels = ['Domain', 'Số URL nguồn', 'ID mẫu', 'URL mẫu', 'Mẫu cũ/mới', 'Vào được?', 'HTTP trang',
              'HTTP robots', 'Lần thử robots', 'Cache hit robots', 'Loại lỗi', 'Lý do', 'Lỗi gốc', 'QC', 'Cờ QC', 'Số ký tự']
    if DIRECT_PUBLIC_PROBE:
        visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'http_status',
                          'error_type', 'error_detail', 'quality_status', 'char_count',
                          'browser_status', 'browser_http_status', 'browser_quality_status', 'browser_char_count', 'browser_error',
                          'access_result', 'effective_engine']
        labels = ['Domain', 'Số URL nguồn', 'ID', 'URL', 'HTTP Requests', 'Lỗi Requests', 'Chi tiết',
                  'QC Requests', 'Ký tự Requests', 'Selenium', 'HTTP Selenium', 'QC Selenium', 'Ký tự Selenium', 'Lỗi Selenium',
                  'Kết quả cuối', 'Engine tải được']
    def esc(value):
        return html.escape(str(value if value is not None else ''), quote=True)
    body = []
    for row in rows:
        cells = []
        for field in visible_fields:
            value = row[field]
            text = (f'<a href="{esc(value)}" target="_blank" rel="noopener noreferrer">{esc(value)}</a>'
                    if field == 'sample_url' else esc(value))
            cells.append('<td>' + text + '</td>')
        body.append(f'<tr data-access="{row["access_result"]}">' + ''.join(cells) + '</tr>')
    style = """<style>body{font:14px/1.5 system-ui,sans-serif;margin:20px;color:#172b4d}
    .table-wrap{overflow:auto;max-height:70vh;border:1px solid #ccd4df}table{border-collapse:collapse;width:100%}
    th,td{padding:9px;border-bottom:1px solid #dce2e9;text-align:left;vertical-align:top}th{background:#e8eef6;position:sticky;top:0}
    td{min-width:95px;max-width:330px;overflow-wrap:anywhere}td:nth-child(4),td:nth-child(9),td:nth-child(10){min-width:260px}
    input,select{padding:8px;margin:0 10px 14px 0}a{color:#1654a0}</style>"""
    table = '<div class="table-wrap"><table><thead><tr>' + ''.join('<th>' + esc(label) + '</th>' for label in labels)
    table += '</tr></thead><tbody>' + ''.join(body) + '</tbody></table></div>'
    heading = f'<h1>Kết quả thử crawl {len(rows)} URL / {len(domains)} hostname</h1><p>Vào được: {counts["VAO_DUOC"]} · Không vào được: {counts["KHONG_VAO_DUOC"]} · Chưa chạy: {counts["CHUA_CHAY"]}</p>'
    heading += '<p>Vào được tính cả Requests và Selenium phục hồi; trang challenge được loại. Các cột Requests vẫn giữ lỗi gốc, QC là kết quả riêng. Kết quả chỉ áp dụng cho các URL mẫu.</p>'
    controls = '<input id="search" placeholder="Tìm domain, URL hoặc lỗi" aria-label="Tìm kiếm"><select id="access" aria-label="Kết quả"><option value="">Tất cả</option><option>VAO_DUOC</option><option>KHONG_VAO_DUOC</option><option>CHUA_CHAY</option></select><span id="count"></span>'
    priority_fields = ['priority_rank', 'domain', 'dataset_url_count', 'selected_urls', 'completed_urls', 'pending_urls', 'access_failures', 'content_issues', 'error_types']
    priority_labels = ['Ưu tiên', 'Domain', 'Số URL dataset', 'Mẫu', 'Đã chạy', 'Chưa chạy', 'Lỗi truy cập', 'Lỗi nội dung', 'Loại lỗi']
    priority_heading = '<h2>Domain còn lỗi — số URL dataset giảm dần</h2><p>Gồm lỗi truy cập và lỗi nội dung/QC. Số URL nguồn không phải số URL đã xác minh lỗi.</p>'
    priority_table = '<div class="table-wrap"><table id="priority"><thead><tr>' + ''.join('<th>'+esc(label)+'</th>' for label in priority_labels)
    priority_table += '</tr></thead><tbody>' + ''.join('<tr>' + ''.join('<td>'+esc(r[f])+'</td>' for f in priority_fields) + '</tr>' for r in priority) + '</tbody></table></div>'
    (RUN_DIR / 'uu_tien_fix_domain.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Ưu tiên domain lỗi</title>'+style+priority_heading+priority_table+'</html>', encoding='utf-8')
    javascript = """<script>const rs=[...document.querySelectorAll('tbody tr')],s=document.getElementById('search'),a=document.getElementById('access');
    function filter(){let n=0;for(const r of rs){const ok=r.textContent.toLowerCase().includes(s.value.toLowerCase())&&(!a.value||r.dataset.access===a.value);r.hidden=!ok;n+=ok?1:0}document.getElementById('count').textContent=n+' hostname'}
    s.addEventListener('input',filter);a.addEventListener('change',filter);filter();</script>"""
    javascript = javascript.replace("document.querySelectorAll('tbody tr')", "document.querySelectorAll('tr[data-access]')").replace("n+' hostname'", "n+' URL'")
    (RUN_DIR / 'ket_qua.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Kết quả crawl domain</title>' + style + heading + priority_heading + priority_table + '<h2>Từng URL mẫu</h2>' + controls + table + javascript + '</html>', encoding='utf-8')
    # Archive outside the run directory to avoid including the archive itself.
    zip_path = Path(shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
    if show:
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        display(HTML(style + priority_heading + priority_table))
        display(HTML(style + heading + table))
        print('File kết quả:', RUN_DIR)
        print('ZIP:', zip_path)
    return rows, summary, zip_path

# Create reports before networking, so partial runs have explicit CHUA_CHAY rows.
rows, summary, zip_path = save_outputs(show=False)
print('Đã chuẩn bị báo cáo và phân loại lỗi.')


## 5. Tải trang và đối chiếu Selenium

Chạy Requests trước, từng URL tuần tự trong một domain; các domain chạy song song.
Mỗi URL có timeout connect/read và ngân sách riêng, không tiêu hao ngân sách vì các mẫu cùng domain chờ nhau.
Ngân sách được kiểm tra giữa các bước; DNS và thử nhiều IP có thể làm thời gian thực vượt mức đặt.
Theo `BROWSER_TEST_MODE`, Selenium thử lại URL Requests lỗi hoặc HTTP 200 nhưng text rỗng/QC lỗi;
có thể đặt `off` để chỉ chạy Requests. Nếu bật Selenium, Chrome được kiểm tra trước lượt Requests.
Trên Linux, notebook tự cài thư viện Chrome còn thiếu bằng apt (gồm ATK); Selenium Manager tìm/tải
Chrome và driver. Preflight thất bại sẽ dừng ô crawl, lưu `browser_setup_error.json`,
`browser_runtime.json`, `browser_preflight.json` và log Chrome để chẩn đoán.
Requests và Selenium có kết quả riêng; DOM trống/challenge không được ghi thành tải được bài.
Requests retry tối đa 3 lần; lỗi mạng được đóng pool kết nối cũ trước lần thử tiếp theo.
Selenium dùng User-Agent thực và retry timeout tối đa 2 lần; 403/challenge không được retry liên tục.
`access_result` tính cả trường hợp Selenium phục hồi, còn các cột lỗi Requests vẫn giữ nguyên.
Kết quả cuối chọn engine có QC tốt hơn; kết quả tương đương giữ Requests.
Parser 120ask lấy câu hỏi/câu trả lời; cnkang lấy thân bài, bỏ menu/bài liên quan.
`summary.json` tách số URL phục hồi truy cập và số URL cải thiện nội dung nhờ Selenium.
Để thử lại riêng nhóm lỗi, đặt `RECOVERY_RUN_DIR` là đường dẫn run cũ và giữ `RESUME_RUN_DIR = None`.
Recovery tạo run mới, kiểm tra hash raw 2xx trước khi reuse, để các checkpoint lỗi được tải lại.
Để chạy lại toàn bộ bằng cấu hình mới, giữ `RESUME_RUN_DIR = None` và Run All.

Mặc định `failed_only` tự chuyển mọi lỗi Requests/trích xuất, text rỗng và QC `review`/`rejected`
sang Selenium, kể cả PDF cần kiểm tra lại. `off` chủ động tắt đối chiếu Selenium;
robots bị chặn và URL không hợp lệ/địa chỉ không được phép vẫn được giữ nguyên.


In [ ]:
import argparse
import os
import socket
import time

# DNS check is diagnostic only; failure here does not stop the crawl.
network_path = RUN_DIR / 'network_check.json'
if not network_path.exists():
    network = {
        'checked_at': datetime.now(timezone.utc).isoformat(),
        'environment': 'kaggle' if Path('/kaggle/working').is_dir() else 'colab' if Path('/content').is_dir() else 'local',
        'python': sys.version, 'platform': sys.platform,
        'proxy_environment_present': {key: bool(os.environ.get(key)) for key in
                                      ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY')},
        'dns': [],
    }
    for domain in NETWORK_CHECK_DOMAINS:
        start = time.monotonic()
        entry = {'domain': domain}
        try:
            entry['addresses'] = sorted({r[4][0] for r in socket.getaddrinfo(domain, 443)})
        except OSError as exc:
            entry.update(exception_class=type(exc).__name__, error=str(exc))
        entry['elapsed_seconds'] = round(time.monotonic() - start, 3)
        network['dns'].append(entry)
    network_path.write_text(json.dumps(network, ensure_ascii=False, indent=2), encoding='utf-8')
print('Network check:', network_path)

try:
    if DIRECT_PUBLIC_PROBE:
        if BROWSER_TEST_MODE != 'off':
            priority_probe.browser_preflight(crawler, RUN_DIR, user_agent=BROWSER_USER_AGENT,
                                             chrome_binary=CHROME_BINARY, load_timeout=BROWSER_LOAD_TIMEOUT)
        if RECOVERY_RUN_DIR:
            recovery = priority_probe.seed_successful_requests(crawler, RUN_DIR, SAMPLES, RECOVERY_RUN_DIR)
            print('Recovery:', len(recovery['reused_ids']), 'URL reuse raw;', len(recovery['retry_ids']), 'URL thử lại.')
        records = priority_probe.crawl_requests(
            crawler, RUN_DIR, SAMPLES, workers=WORKERS, delay=DELAY,
            attempts=PAGE_ATTEMPTS, user_agent=USER_AGENT,
            timeout=(PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT), budget=PAGE_REQUEST_BUDGET, max_new=MAX_NEW, log_every=LOG_EVERY)
        browser_samples = priority_probe.select_browser_samples(SAMPLES, records, BROWSER_TEST_MODE)
        if browser_samples:
            priority_probe.crawl_browser(crawler, RUN_DIR, browser_samples,
                                        user_agent=BROWSER_USER_AGENT, chrome_binary=CHROME_BINARY, delay=DELAY,
                                        load_timeout=BROWSER_LOAD_TIMEOUT, attempts=BROWSER_ATTEMPTS, log_every=LOG_EVERY)
        crawler.report(argparse.Namespace(out=str(RUN_DIR)))
    else:
        crawler.locked_crawl(argparse.Namespace(
            out=str(RUN_DIR), workers=WORKERS, delay=DELAY, attempts=ATTEMPTS, max_new=MAX_NEW,
            log_every=LOG_EVERY,
            robots_attempts=ROBOTS_ATTEMPTS, robots_connect_timeout=ROBOTS_CONNECT_TIMEOUT,
            robots_read_timeout=ROBOTS_READ_TIMEOUT, robots_retry_budget=ROBOTS_RETRY_BUDGET,
            robots_failure_ttl=ROBOTS_FAILURE_TTL,
        ))
finally:
    rows, summary, zip_path = save_outputs()


## 6. Xem lỗi riêng và tải output

`khong_vao_duoc.csv` chứa domain, URL, loại lỗi, lý do tiếng Việt và lỗi gốc.
`loi_theo_loai.csv` gom số hostname và số URL theo từng lỗi (HTTP 403, timeout,
robots trả HTML, robots cấm crawl, DNS, SSL…). File JSON trong `records/`
giữ trace request/redirect; `robots/` giữ bằng chứng robots.txt.
`network_check.json` ghi DNS của mọi hostname trong bộ mẫu để đối chiếu Kaggle/local;
`priority_probe_config.json` ghi User-Agent/header, timeout và chế độ tải trang.
`requests_vs_selenium.csv` và `browser_records/` giữ kết quả trình duyệt riêng;
`browser_setup_error.json` được ghi nếu Chrome/driver không khởi động được.
`browser_runtime.json`, `browser_preflight.json`, `chrome_dependencies.log` và `chromedriver.log`
ghi kiểm tra môi trường/cài thư viện/khởi động Chrome (log cài apt chỉ có khi cần cài thêm).
ZIP gồm kết quả, raw, ảnh chụp và snapshot để kiểm tra riêng.

Kaggle: tải ZIP trong **Output**. Colab: ô dưới hiện nút tải ZIP.
Jupyter local: dùng các liên kết file hoặc mở đường dẫn output được in ra.


In [ ]:
# Có thể chạy lại ô này để xuất báo cáo từ checkpoint mà không crawl thêm.
rows, summary, zip_path = save_outputs(show=False)
failed_rows = [r for r in rows if r['access_result'] == 'KHONG_VAO_DUOC']
print(f'Không vào được: {len(failed_rows)}/{len(rows)} URL, thuộc {len({r["domain"] for r in failed_rows})} hostname')
for row in failed_rows:
    print(f"{row['domain']} | {row['error_type']} | {row['error_detail']}")
print('\nOutput:', RUN_DIR)
for name in ['WEB_ERRORS.md', 'uu_tien_fix_domain.html', 'uu_tien_fix_domain.csv', 'tong_hop_domain.csv',
             'requests_vs_selenium.csv',
             'ket_qua.html', 'ket_qua.csv', 'vao_duoc.csv', 'khong_vao_duoc.csv',
             'loi_theo_loai.csv', 'can_xem_noi_dung.csv', 'chua_chay.csv']:
    display(FileLink(str(RUN_DIR / name)))
display(FileLink(str(zip_path)))

if 'google.colab' in sys.modules:
    import ipywidgets as widgets
    from google.colab import files
    button = widgets.Button(description='Tải ZIP kết quả')
    button.on_click(lambda _: files.download(str(zip_path)))
    display(button)
